# Development CPU metric receivers
CPU only: verify completed Kaggle shards through the official dispatcher and recompute saved forecasts. Immutable routing, one submission per reader, bounded CPU relays. No training, GPU, Final or retry on provider failure.


In [ ]:
import hashlib
QUEUE_SOURCE='"""Build bounded private Kaggle Development jobs and a CPU-only relay.\n\nUses Kaggle\'s existing notebook identity. No credentials, paid scaling, Final\nlabels or local work are included. Submission failures are never retried.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nimport zipfile\nfrom pathlib import Path\n\nOWNER = "hungluongduc"\nPAYLOAD = f"{OWNER}/hung-matched-payload-44b8532-20261027-v1/1"\nALLOCATION_SHA = "12f5bc9ee402a806faedc89574a4e00c427c384335a3868668adf8ce7f93a0f2"\nGROUPS = (\n    (2, 3),\n    tuple(range(8, 14)),\n    tuple(range(14, 20)),\n    tuple(range(20, 26)),\n    (26, 27, 28),\n    (33, 34),\n    (35, 36),\n)\nLOCAL = {4, 5, 6, 7, 29, 30, 31, 32, 37, 38, 39}\nSTORAGE_SOURCE = ""  # Embedded from the versioned storage module when building v2.\nEXPECTED_JOBSET_SHA = {\n    "shard_0001": "e2051de08dfb0f0ee91c3617151f08d5cd2c712bb88344244980e4ec2f350c69",\n    "shard_0002": "1cf89024e9c8b9d8488b386a54a0995924fad19a9334f9539d50843d345260f1",\n    "shard_0003": "145d75ce5a847f1494d7ec4285e47812c769acb7206bc0da7a20a48e4545f4d8",\n    "shard_0004": "a7e50150565ca83055e33a9b246e6f8ae7fea6b376d4e1cde6ae55ec0aa8192a",\n    "shard_0005": "0912c4c7c32350f611bce7493b207cba24810e4aaa20d793a7ee8f2769e1990c",\n    "shard_0006": "50b870631b6c422dc5b2a29c5e71288a1c07de79428e36cadd989d0a764d1a28",\n    "shard_0007": "7b448f6b347ad6feae86b5301d97ad4e14a5d590ef70f24c0e6a877149144ec5",\n    "shard_0008": "928996ffad7ba48f210d2b69cc23cdf4c402a8d34c9a79be76a58a1591d7ff9d",\n    "shard_0009": "ac51753e9c13f2e6f0677fda3e1e67174fb8a2080a7a859e02465a0c790278f0",\n    "shard_0010": "9ec71c246ca5334cc5a622cbff9e7b76e6535e447ce0ae6520a8ee9095dc34a3",\n    "shard_0011": "7ff773c49768e8f73a3b7245e6b6a178de0f599adccee87234f2c4dfa2299932",\n    "shard_0012": "ae3f9b967f922de325bd35c045eeaafddb3647da47c09d2749887c0351102017",\n    "shard_0013": "d19ce730e5d2bd6d02c545359838569e1b712269689a58f9ca0e1a03d30b18b8",\n    "shard_0014": "1454cb90a5e8e0406617e4a0a8a1408040ae570ed6ce9cf81479e08878b5b6a8",\n    "shard_0015": "e839173725dde0f0e4fb122a9813fac263907e08dd9889d2011073e1e52c30d6",\n    "shard_0016": "ad6e9fe1219eec06bcde8a95ef333f50bee40c27ed97459375d208e76a01ed0f",\n    "shard_0017": "a494de718f33adde057a68ef1b4e9e0ea31b56461594f9316fdb6e45211fee75",\n    "shard_0018": "05c45a5b8bed8e143d16053d892c8abede23633e6c7d96d41288b601f429df20",\n    "shard_0019": "15761f42e1b901f0b99f98d90a61f9c9f81ba058535054f612be11867048bfef",\n    "shard_0020": "15e41973e7f727fdcbe7f239922ad02dc05c8efe77cc9609245e29f6278be493",\n    "shard_0021": "14e4fc0a128f8ed81e52b34d15af9a123503e0d5903f65eb16b5a20c477de5cb",\n    "shard_0022": "229dfba032263815967bfa18f3a5baef677747ed0c6f828e89202c12703e78fc",\n    "shard_0023": "f9def5fc4236ed9d9a540c937d554795b4da684d6b0705225dc8dff0f7f01c69",\n    "shard_0024": "86cb50e120c3081676b0e75a13055659ab17c7092ea8473e311fadad1328ce00",\n    "shard_0025": "ffc9c9a46dc12a5f843182e1e092c2e434cab2432d218ae1076b987c3b3e1937",\n    "shard_0026": "3cc709041d096465692797a24d8290c6b8d2e14f3e994d50a734c8fe34e64646",\n    "shard_0027": "19f14ba77bf687edcf8847933ea5807006096d9a8eb0128560ad20f1cddc5369",\n    "shard_0028": "a1328e31335db38f13e20e843025c25e6767ed1e86e11ff39b1da10a994db567",\n    "shard_0029": "33c10aa43b9d1c2675b4cc89e087a12e6c8875c5cd0ac5789a39775a760e7dfc",\n    "shard_0030": "a908b9eb43e288809722c1c8300370bf0acac961aaaba25d98a169008457395a",\n    "shard_0031": "35d10c724dd8ac4ed4bb03f864c35288f9edc453900024e12383a328d0c6a0d9",\n    "shard_0032": "b6fc0d6d0924f2c970da1e40783c6f9ef740d5898458c7caffb7c66830659270",\n    "shard_0033": "a570c5e215f60b6463e4641f73ce13f7035552162a851d754f524d754031debe",\n    "shard_0034": "79b8e55f8a74e34b504a736555e9add02e1422bbd5a872509bdab00ae4a21d8a",\n    "shard_0035": "0ecdc975791fe6b62b0d1be6514d5ca243b3892bf8e8fe612be00dd71a93dac0",\n    "shard_0036": "c0f24e5038e361798c83d44bd292ef8d099cf575ba00fcab92d55cf178873c2a",\n    "shard_0037": "aee8e665e66576f461c3a69de62ede64c0840c1189c8e26b51c158cd3b2303ce",\n    "shard_0038": "09b5c62ae11c1563ee12d6d97c9e10902a48da50a89d80129b581721b35606f2",\n    "shard_0039": "4177fe39acd6ac291e38b82aea9cc148376f578a58de1bf2d875ed9fa8dddab4",\n}\n\n\nEXPECTED_PROVIDER_PINS = {\n    "shard_0002": (\n        "70dff1c41380aa781062470b6986829e470e320a1cd0522ac49c63956ea6d12d",\n        "7d99e722c6e27b7257a709182a0827b63c4a1599cfe14f3472e9e4a914cf8ad1",\n    ),\n    "shard_0003": (\n        "70dff1c41380aa781062470b6986829e470e320a1cd0522ac49c63956ea6d12d",\n        "7d99e722c6e27b7257a709182a0827b63c4a1599cfe14f3472e9e4a914cf8ad1",\n    ),\n    "shard_0008": (\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\n    ),\n    "shard_0009": (\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\n    ),\n    "shard_0010": (\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\n    ),\n    "shard_0011": (\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\n    ),\n    "shard_0012": (\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\n    ),\n    "shard_0013": (\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\n    ),\n    "shard_0014": (\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\n    ),\n    "shard_0015": (\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\n    ),\n    "shard_0016": (\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\n    ),\n    "shard_0017": (\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\n    ),\n    "shard_0018": (\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\n    ),\n    "shard_0019": (\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\n    ),\n    "shard_0020": (\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\n    ),\n    "shard_0021": (\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\n    ),\n    "shard_0022": (\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\n    ),\n    "shard_0023": (\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\n    ),\n    "shard_0024": (\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\n    ),\n    "shard_0025": (\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\n    ),\n    "shard_0026": (\n        "2e9faad307c29ca681eedcc0f64da2ab6ec00ef3893a280d1aead08b30d16836",\n        "7750cfc08778c747122a2a2e82190a63c5e2f034d68f690748290c599b7c41be",\n    ),\n    "shard_0027": (\n        "2e9faad307c29ca681eedcc0f64da2ab6ec00ef3893a280d1aead08b30d16836",\n        "7750cfc08778c747122a2a2e82190a63c5e2f034d68f690748290c599b7c41be",\n    ),\n    "shard_0028": (\n        "2e9faad307c29ca681eedcc0f64da2ab6ec00ef3893a280d1aead08b30d16836",\n        "7750cfc08778c747122a2a2e82190a63c5e2f034d68f690748290c599b7c41be",\n    ),\n    "shard_0033": (\n        "a15728a0c6e54de080e0fbfce2b49fbae9cfeedeaa2a76f281f584f369847886",\n        "0a02b5c6737445748fcff41d88b2a44cacdfd1f394d80f0a341de47bd983e08b",\n    ),\n    "shard_0034": (\n        "a15728a0c6e54de080e0fbfce2b49fbae9cfeedeaa2a76f281f584f369847886",\n        "0a02b5c6737445748fcff41d88b2a44cacdfd1f394d80f0a341de47bd983e08b",\n    ),\n    "shard_0035": (\n        "19129a69120e4f1cfd7dc6eae08d9a5ce4a20373fd9f48f584d8faf04801cb7d",\n        "bd9bc74c3c417f04fbb16781f9c3fb33b07b78d193145dfeae953e7ac610a01f",\n    ),\n    "shard_0036": (\n        "19129a69120e4f1cfd7dc6eae08d9a5ce4a20373fd9f48f584d8faf04801cb7d",\n        "bd9bc74c3c417f04fbb16781f9c3fb33b07b78d193145dfeae953e7ac610a01f",\n    ),\n}\n\n\ndef digest(path: Path) -> str:\n    result = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef validate_groups() -> None:\n    flat = [n for group in GROUPS for n in group]\n    assert len(flat) == len(set(flat)) and not set(flat) & LOCAL\n    assert set(flat) | {1} == {1, 2, 3, *range(8, 29), 33, 34, 35, 36}\n    assert all(0 < len(group) <= 6 for group in GROUPS)\n\n\ndef slug(index: int) -> str:\n    assert 0 <= index < len(GROUPS)\n    return f"hung-matched-group{index + 2:02d}-44b8532-20261027" + ("-v2" if index else "")\n\n\ndef verify_predecessor(\n    root: Path, shards: tuple[int, ...], *, archive_name="artifacts.zip"\n) -> list[dict]:\n    """Hash/CRC readback of a pinned predecessor version, before new training."""\n    records = []\n    for number in shards:\n        name = f"shard_{number:04d}"\n        matches = [\n            p\n            for p in root.rglob("receipt.json")\n            if json.loads(p.read_text()).get("shard_id") == name\n            and p.with_name(archive_name).is_file()\n        ]\n        assert len(matches) == 1, f"Missing/ambiguous predecessor {name}"\n        receipt = json.loads(matches[0].read_text())\n        archive = matches[0].with_name(archive_name)\n        assert receipt["shard_id"] == name\n        assert receipt["exit_code"] == receipt["local_verify_exit_code"] == 0\n        assert (receipt["plan_sha256"], receipt["shards_sha256"]) == EXPECTED_PROVIDER_PINS[name]\n        jobset = json.dumps(sorted(receipt["job_ids"]), separators=(",", ":")).encode()\n        assert hashlib.sha256(jobset).hexdigest() == EXPECTED_JOBSET_SHA[name]\n        assert archive.stat().st_size == receipt["artifact_bytes"]\n        assert digest(archive) == receipt["artifact_sha256"]\n        with zipfile.ZipFile(archive) as contents:\n            names = contents.namelist()\n            assert len(names) == len(set(names))\n            assert all(not n.startswith(("/", "\\\\")) and ".." not in n.split("/") for n in names)\n            assert contents.testzip() is None\n            identity = json.loads(contents.read("dispatch_identity.json"))\n            assert identity["plan_sha256"] == receipt["plan_sha256"]\n            measured = {\n                n.split("/")[1]\n                for n in names\n                if n.startswith("jobs/") and n.endswith("/dispatch_measurement.json")\n            }\n            assert set(measured) == set(receipt["job_ids"])\n            assert len(receipt["job_ids"]) == len(set(receipt["job_ids"])) == 20\n        records.append(\n            {\n                "shard": name,\n                "sha256": receipt["artifact_sha256"],\n                "bytes": receipt["artifact_bytes"],\n                "crc": "passed",\n            }\n        )\n    return records\n\n\ndef status(slug_name: str) -> str:\n    from kagglehub.clients import build_kaggle_client\n    from kagglesdk.kernels.types.kernels_api_service import ApiGetKernelSessionStatusRequest\n\n    request = ApiGetKernelSessionStatusRequest()\n    request.user_name, request.kernel_slug = OWNER, slug_name\n    with build_kaggle_client() as client:\n        response = client.kernels.kernels_api_client.get_kernel_session_status(request)\n    value = response.status.name\n    assert value in {\n        "QUEUED",\n        "RUNNING",\n        "COMPLETE",\n        "ERROR",\n        "NEW_SCRIPT",\n        "CANCEL_REQUESTED",\n        "CANCEL_ACKNOWLEDGED",\n    }, value\n    return value\n\n\ndef submit(\n    name: str,\n    source: str,\n    *,\n    gpu: bool,\n    predecessor: str | None = None,\n    datasets: list[str] | None = None,\n) -> dict:\n    """One submission attempt. Caller must hold the unique relay chain position."""\n    from kagglehub.clients import build_kaggle_client\n    from kagglesdk.kernels.types.kernels_api_service import ApiSaveKernelRequest\n\n    claim(name + "-submit", {"source_sha256": hashlib.sha256(source.encode()).hexdigest()})\n    request = ApiSaveKernelRequest()\n    request.slug, request.new_title = f"{OWNER}/{name}", name\n    request.text, request.language, request.kernel_type = source, "python", "script"\n    request.is_private, request.enable_gpu, request.enable_internet = True, gpu, True\n    request.session_timeout_seconds = 8 * 3600 if gpu else 5 * 3600\n    request.dataset_data_sources = (\n        datasets if datasets is not None else ([PAYLOAD] if gpu or predecessor else [])\n    )\n    request.kernel_data_sources = [f"{OWNER}/{predecessor}/1"] if predecessor else []\n    if gpu:\n        request.machine_shape = "NvidiaTeslaT4"\n    with build_kaggle_client() as client:\n        response = client.kernels.kernels_api_client.save_kernel(request)\n    assert not response.error and response.version_number == 1, "Unexpected version; stop"\n    assert not response.invalid_dataset_sources and not response.invalid_kernel_sources\n    record = {\n        "name": name,\n        "url": response.url,\n        "version": response.version_number,\n        "gpu": gpu,\n        "source_sha256": hashlib.sha256(source.encode()).hexdigest(),\n    }\n    print(json.dumps({"submitted": record}), flush=True)\n    return record\n\n\ndef claim(name: str, record: dict) -> None:\n    """Create-only private Dataset claim; 409 never becomes a new version."""\n    from kagglehub.clients import build_kaggle_client\n    from kagglehub.gcs_upload import upload_files_and_directories\n    from kagglesdk.blobs.types.blob_api_service import ApiBlobType\n    from kagglesdk.datasets.types.dataset_api_service import ApiCreateDatasetRequest\n\n    folder = Path("/kaggle/working/claims") / name\n    folder.mkdir(parents=True, exist_ok=False)\n    (folder / "claim.json").open("x").write(\n        json.dumps({**record, "allocation_sha256": ALLOCATION_SHA})\n    )\n    upload = upload_files_and_directories(\n        str(folder), item_type=ApiBlobType.DATASET, ignore_patterns=[]\n    ).to_proto()\n    request = ApiCreateDatasetRequest()\n    request.owner_slug, request.slug, request.title = OWNER, name, name\n    request.files, request.directories, request.is_private = upload.files, upload.directories, True\n    with build_kaggle_client() as client:\n        result = client.datasets.dataset_api_client.create_dataset(request)\n    assert not result.error, "Claim failed; no submission/training retry"\n\n\ndef module_prefix(module_source: str) -> str:\n    compile(module_source, "hung_kaggle_queue.py", "exec")\n    return module_source + "\\nMODULE_SOURCE = " + repr(module_source) + "\\n"\n\n\ndef compose_module_source(queue_source: str, storage_source: str) -> str:\n    compile(storage_source, "hung_kaggle_storage.py", "exec")\n    return queue_source + "\\nSTORAGE_SOURCE = " + repr(storage_source) + "\\n"\n\n\ndef gpu_source(module_source: str, setup: str, parity: str, index: int) -> str:\n    assert 1 <= index < len(GROUPS)\n    selected = [f"shard_{n:04d}" for n in GROUPS[index]]\n    base_name = f"hung-matched-group{index + 2:02d}-v1"\n    setup = setup.replace("hung-matched-group02-v1", base_name)\n    setup = setup.replace(\'["shard_0002", "shard_0003"]\', repr(selected))\n    source = module_prefix(module_source)\n    source += "\\nassert STORAGE_SOURCE\\nexec(STORAGE_SOURCE, globals())\\n"\n    source += f"\\nclaim({slug(index)!r}+\'-execute\', {{\'group_index\': {index}}})\\n" + setup\n    source += "\\nreadback_started = time.perf_counter()\\n"\n    source += f"readback = verify_predecessor(Path(\'/kaggle/input\'), {GROUPS[index - 1]!r})\\n"\n    source += "provider.save(base / \'predecessor-readback.json\', {\'records\': readback, \'seconds\': time.perf_counter()-readback_started})\\n"\n    source += "print(json.dumps({\'predecessor_readback\': readback}), flush=True)\\n"\n    source += "\\n" + parity + "\\nassert passed\\n"\n    source += f"selected = {selected!r}\\nreceipts = []\\ngroup_start = time.perf_counter()\\n"\n    source += """\nfor shard_id in selected:\n    from kagglehub.clients import build_kaggle_client\n    from kagglesdk.kernels.types.kernels_api_service import ApiGetAcceleratorQuotaStatisticsRequest\n    with build_kaggle_client() as client:\n        quota = client.kernels.kernels_api_client.get_accelerator_quota_statistics(\n            ApiGetAcceleratorQuotaStatisticsRequest()).gpu_quota\n    assert quota is not None and not quota.is_pay_to_scale_enabled\n    assert all(v is not None for v in (quota.total_time_allowed, quota.time_used, quota.time_reserved))\n    available = (quota.total_time_allowed-quota.time_used-quota.time_reserved).total_seconds()\n    assert available >= 3600 + 20 * 1810.4 / 20 * 1.25, \'Fresh quota + reserve failed\'\n    assert shutil.disk_usage(base).free >= 8*1024**3, \'Disk reserve failed\'\n    assert time.perf_counter()-group_start < 6*3600, \'Bounded session limit\'\n    record = provider.execute_shard(repo, shard_id, base / shard_id)\n    persist_shard(record, base/shard_id/\'artifacts.zip\', module_prefix(MODULE_SOURCE),\n                  owner=OWNER, submit=submit, status=status)\n    receipts.append(record)\n    print(json.dumps(record), flush=True)\nprovider.save(base / \'group-complete.json\', {\'shards\': selected, \'receipts\': receipts,\n    \'allocation_queue_sha256\': ALLOCATION_SHA, \'scope\': \'Development only; peer acceptance pending\'})\nprint(json.dumps({\'group_complete\': True, \'shards\': selected}), flush=True)\n"""\n    compile(source, base_name, "exec")\n    return source\n\n\ndef relay_source(\n    module_source: str, setup: str, parity: str, index: int, ordinal: int, *, submit_current: bool\n) -> str:\n    assert 0 <= index < len(GROUPS) and ordinal < 12\n    return (\n        module_prefix(module_source)\n        + "\\nSETUP = "\n        + repr(setup)\n        + "\\nPARITY = "\n        + repr(parity)\n        + f"\\nrelay(MODULE_SOURCE, SETUP, PARITY, {index}, {ordinal}, "\n        + f"submit_current={submit_current!r})\\n"\n    )\n\n\ndef relay(\n    module_source: str, setup: str, parity: str, index: int, ordinal: int, *, submit_current: bool\n) -> None:\n    """CPU SDK relay: serial fixed GPU groups; 3h CPU relay bounds, no LLM polling."""\n    validate_groups()\n    claim(f"hung-development-relay-{ordinal:02d}-execute-20261027-v2", {"group_index": index})\n    evidence = Path(f"/kaggle/working/relay-{ordinal:02d}")\n    evidence.mkdir(exist_ok=False)\n    started = time.monotonic()\n    if submit_current:\n        assert index > 0\n        submit(\n            slug(index),\n            gpu_source(module_source, setup, parity, index),\n            gpu=True,\n            predecessor=slug(index - 1),\n        )\n    while True:\n        state = status(slug(index))\n        assert state not in {"ERROR", "CANCEL_REQUESTED", "CANCEL_ACKNOWLEDGED", "NEW_SCRIPT"}, (\n            state\n        )\n        if state == "COMPLETE":\n            (evidence / f"group-{index:02d}-terminal.json").open("x").write(\n                json.dumps({"status": state, "version": 1, "name": slug(index)})\n            )\n            if index == len(GROUPS) - 1:\n                # A separate CPU version mounts the last completed Output and checks bytes.\n                proof = (\n                    module_prefix(module_source)\n                    + f"\\nrecords=verify_predecessor(Path(\'/kaggle/input\'), {GROUPS[index]!r})\\n"\n                    + "Path(\'/kaggle/working/final-development-readback.json\').write_text(json.dumps({\'records\':records,\'scope\':\'Development technical readback only; no Final evaluation\'}))\\n"\n                    + "print(json.dumps({\'passed\':True,\'records\':records}),flush=True)\\n"\n                )\n                submit(\n                    "hung-development-last-readback-20261027-v2",\n                    proof,\n                    gpu=False,\n                    predecessor=slug(index),\n                )\n                return\n            index += 1\n            submit(\n                slug(index),\n                gpu_source(module_source, setup, parity, index),\n                gpu=True,\n                predecessor=slug(index - 1),\n            )\n        if time.monotonic() - started >= 3 * 3600:\n            assert ordinal + 1 < 12, "CPU relay count bound"\n            name = f"hung-development-cpu-relay-{ordinal + 1:02d}-20261027-v2"\n            submit(\n                name,\n                relay_source(\n                    module_source, setup, parity, index, ordinal + 1, submit_current=False\n                ),\n                gpu=False,\n            )\n            (evidence / "handoff.json").open("x").write(json.dumps({"next": name, "index": index}))\n            return\n        time.sleep(60)\n'
assert hashlib.sha256(QUEUE_SOURCE.encode()).hexdigest()=='4ee0718f077e587e14619ec2effe1637de93a18750d81da11e8c576a6de63093'
METRIC_SOURCE='"""Finite CPU-only receivers for the existing immutable Kaggle Development route."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\n\nTRIAL = "hung-shard0008-full-cpu-restore-20261027-v1"\nMODEL_CODE = "44b8532959d76fb20691881539f991ca44983727"\nPAYLOAD_SHA = "23a6d420523d990d07ff07d814ed642a851925283a102a3ae806f3b666e1d6f4"\nHELPER_SHA = "cadcd80d671d3b49a317312f125735362ffba3068247f1714a0ae56c6be83424"\n\n\ndef bundle_sources(queue: str, metric_queue: str, receiver: str, scorer: str) -> str:\n    """Compile separate versioned modules, preserving future-import boundaries."""\n    sources = {\n        "QUEUE_SOURCE": queue,\n        "METRIC_SOURCE": metric_queue,\n        "RECEIVER_SOURCE": receiver,\n        "SCORER_SOURCE": scorer,\n    }\n    result = "import hashlib\\n"\n    for name, source in sources.items():\n        compile(source, name, "exec")\n        result += f"{name}={source!r}\\n"\n        result += f"assert hashlib.sha256({name}.encode()).hexdigest()=={hashlib.sha256(source.encode()).hexdigest()!r}\\n"\n    result += "exec(compile(QUEUE_SOURCE,\'pinned_queue\',\'exec\'),globals())\\n"\n    result += "exec(compile(METRIC_SOURCE,\'pinned_metric_queue\',\'exec\'),globals())\\n"\n    compile(result, "cpu-metric-bundle", "exec")\n    return result\n\n\ndef readers(groups: tuple, provider_pins: dict) -> list[dict]:\n    """Only allocated Kaggle shards; the already-running shard8 trial is reused."""\n    result = [\n        {\n            "shard_id": "shard_0001",\n            "training": "hung-matched-shard0001-44b8532-20261027",\n            "base": "hung-matched-shard0001-v1",\n            "plan_sha256": "80a1666cf2083594c9a2bc0b0a4dc3e30118eda43d07187cd5b6d8b51f90e0bb",\n            "partition_sha256": "e2c6cb6f15a7473c6cdfd7912fefa838d63a770cf352dff93c2cccc3ae86a15c",\n        }\n    ]\n    for index, group in enumerate(groups):\n        training = (\n            "hung-matched-group03-recovery-20261027-v2"\n            if index == 1\n            else f"hung-matched-group{index + 2:02d}-44b8532-20261027" + ("-v2" if index else "")\n        )\n        for number in group:\n            if number == 8:\n                continue\n            shard = f"shard_{number:04d}"\n            plan_sha, partition_sha = provider_pins[shard]\n            result.append(\n                {\n                    "shard_id": shard,\n                    "training": training,\n                    "base": f"hung-matched-group{index + 2:02d}-v1",\n                    "plan_sha256": plan_sha,\n                    "partition_sha256": partition_sha,\n                }\n            )\n    expected = {1, 2, 3, *range(9, 29), 33, 34, 35, 36}\n    assert {int(r["shard_id"][6:]) for r in result} == expected and len(result) == len(expected)\n    return result\n\n\ndef reader_name(record: dict) -> str:\n    return f"hung-cpu-metrics-{record[\'shard_id\'].replace(\'_\', \'\')}-20261027-v1"\n\n\ndef owned_kernel_exists(slug_name: str) -> bool:\n    """Observe owned resources before asking status of a not-yet-submitted group."""\n    from kagglehub.clients import build_kaggle_client\n    from kagglesdk.kernels.types.kernels_api_service import ApiListKernelsRequest\n    from kagglesdk.kernels.types.kernels_enums import KernelsListViewType\n\n    request = ApiListKernelsRequest()\n    request.user, request.page_size = "hungluongduc", 100\n    request.group = KernelsListViewType.PUBLIC_AND_USERS_PRIVATE\n    seen = set()\n    with build_kaggle_client() as client:\n        for _ in range(10):\n            result = client.kernels.kernels_api_client.list_kernels(request)\n            if any(k.ref == f"hungluongduc/{slug_name}" for k in result.kernels):\n                return True\n            if not result.next_page_token:\n                return False\n            assert result.next_page_token not in seen, "Owned inventory pagination repeated"\n            seen.add(result.next_page_token)\n            request.page_token = result.next_page_token\n    raise ValueError("Owned inventory bound exceeded; no status/submission inference")\n\n\ndef receiver_source(record: dict, receiver: str, scorer: str) -> str:\n    import hashlib\n\n    source = """import hashlib, importlib.util, json, types\nfrom pathlib import Path\nimport torch\nassert not torch.cuda.is_available(), \'CPU receiver only\'\n"""\n    source += f"SPEC={record!r}\\nRECEIVER_SOURCE={receiver!r}\\nSCORER_SOURCE={scorer!r}\\n"\n    source += f"MODEL_CODE={MODEL_CODE!r}\\nPAYLOAD_SHA={PAYLOAD_SHA!r}\\nHELPER_SHA={HELPER_SHA!r}\\n"\n    source += f"RECEIVER_SHA={hashlib.sha256(receiver.encode()).hexdigest()!r}\\n"\n    source += f"SCORER_SHA={hashlib.sha256(scorer.encode()).hexdigest()!r}\\n"\n    source += """\npayloads=list(Path(\'/kaggle/input\').rglob(\'provider-payload.zip.bin\'))\nassert len(payloads)==1, \'Pinned payload missing/ambiguous\'\nhelper=payloads[0].parent/\'hung_provider.py\'\nassert hashlib.sha256(helper.read_bytes()).hexdigest()==HELPER_SHA\nmodule_spec=importlib.util.spec_from_file_location(\'hung_provider\',helper)\nprovider=importlib.util.module_from_spec(module_spec); module_spec.loader.exec_module(provider)\nreceipts=[p for p in Path(\'/kaggle/input\').rglob(\'receipt.json\')\n          if p.with_name(\'artifacts.zip\').is_file()\n          and json.loads(p.read_bytes()).get(\'shard_id\')==SPEC[\'shard_id\']]\nassert len(receipts)==1, \'Canonical completed shard missing/ambiguous\'\nrepo,plan=provider.materialize(payloads[0],PAYLOAD_SHA,Path(\'/kaggle/working\')/SPEC[\'base\'])\nassert provider.sha256(plan)==SPEC[\'plan_sha256\']\nassert provider.sha256(repo/\'.cache/provider/shards.json\')==SPEC[\'partition_sha256\']\nassert hashlib.sha256(RECEIVER_SOURCE.encode()).hexdigest()==RECEIVER_SHA\nassert hashlib.sha256(SCORER_SOURCE.encode()).hexdigest()==SCORER_SHA\nreceiver=types.ModuleType(\'pinned_cloud_receiver\')\nexec(compile(RECEIVER_SOURCE,\'pinned_cloud_receiver\',\'exec\'),receiver.__dict__)\nscorer={\'__name__\':\'pinned_saved_forecast_scorer\'}\nexec(compile(SCORER_SOURCE,\'pinned_saved_forecast_scorer\',\'exec\'),scorer)\nsummary=receiver.receive(provider,scorer,receipts[0].with_name(\'artifacts.zip\'),receipts[0],repo,\n                         expected_code=MODEL_CODE,evidence=repo.parent/\'cpu-readback\',\n                         scorer_sha256=SCORER_SHA)\nassert summary[\'passed\'] and summary[\'executed\'] is False and summary[\'jobs\']==20\nsummary.update({\'receiver_sha256\':RECEIVER_SHA,\'transport_helper_sha256\':HELPER_SHA,\n                \'derivation\':json.loads((repo.parent/\'derivation.json\').read_bytes()),\n                \'device\':\'cpu\',\'torch\':torch.__version__})\nassert summary[\'derivation\'][\'parent_plan_sha256\']==\'2d448d2584f65baf7e09445e013b4283314b08ea2cce855c6850e9fc12ba87ee\'\nprovider.save(repo.parent/\'receiver-proof.json\',summary)\nprint(json.dumps({\'receiver_proof\':summary}),flush=True)\n"""\n    compile(source, "cpu-shard-receiver", "exec")\n    return source\n\n\ndef relay_source(\n    bundle: str, records: list[dict], index: int, ordinal: int, submitted: bool\n) -> str:\n    source = bundle + f"\\nMETRIC_BUNDLE={bundle!r}\\nMETRIC_READERS={records!r}\\n"\n    source += (\n        "assert owned_kernel_exists(METRIC_READERS[0][\'training\']), \'Owned inventory capability gate failed\'\\n"\n        "print(json.dumps({\'owned_inventory_capability\':\'passed\',\'gpu\':False}),flush=True)\\n"\n    )\n    source += (\n        f"run_queue(METRIC_BUNDLE,METRIC_READERS,{index},{ordinal},{submitted!r},"\n        "submit=submit,status=status,claim=claim,exists=owned_kernel_exists,receiver=RECEIVER_SOURCE,scorer=SCORER_SOURCE)\\n"\n    )\n    compile(source, "cpu-receiver-relay", "exec")\n    return source\n\n\ndef run_queue(\n    bundle: str,\n    records: list[dict],\n    index: int,\n    ordinal: int,\n    submitted: bool,\n    *,\n    submit,\n    status,\n    claim,\n    exists,\n    receiver: str,\n    scorer: str,\n) -> None:\n    """Only provider status handles drive handoffs; no GPU or submission retry."""\n    assert 0 <= index < len(records) and ordinal < 12\n    assert len({r["shard_id"] for r in records}) == len(records)\n    claim(f"hung-cpu-metrics-relay-{ordinal:02d}-execute-20261027-v1", {"reader_index": index})\n    root = Path(f"/kaggle/working/cpu-metrics-relay-{ordinal:02d}")\n    root.mkdir(exist_ok=False)\n    started = time.monotonic()\n    known_training = set()\n    while True:\n        trial = status(TRIAL)\n        assert trial in {"QUEUED", "RUNNING", "COMPLETE"}, f"CPU trial failed: {trial}"\n        record = records[index]\n        if trial == "COMPLETE":\n            if not submitted:\n                name = record["training"]\n                if name not in known_training and exists(name):\n                    known_training.add(name)\n                training = status(name) if name in known_training else "NOT_OBSERVED"\n                assert training in {"NOT_OBSERVED", "QUEUED", "RUNNING", "COMPLETE"}, (\n                    f"Provider failed: {training}"\n                )\n                if training == "COMPLETE":\n                    submit(\n                        reader_name(record),\n                        receiver_source(record, receiver, scorer),\n                        gpu=False,\n                        predecessor=record["training"],\n                    )\n                    submitted = True\n            if submitted:\n                state = status(reader_name(record))\n                assert state in {"QUEUED", "RUNNING", "COMPLETE"}, f"CPU receiver failed: {state}"\n                if state == "COMPLETE":\n                    proof = {\n                        "shard_id": record["shard_id"],\n                        "cpu_reader": reader_name(record),\n                        "version": 1,\n                        "status": state,\n                        "scope": "Development technical readback only",\n                    }\n                    with (root / (record["shard_id"] + ".json")).open(\n                        "x", encoding="utf-8"\n                    ) as stream:\n                        json.dump(proof, stream)\n                    print(json.dumps({"completed_cpu_receiver": proof}), flush=True)\n                    index += 1\n                    submitted = False\n                    if index == len(records):\n                        with (root / "complete.json").open("x", encoding="utf-8") as stream:\n                            json.dump({"receivers": len(records)}, stream)\n                        return\n        if time.monotonic() - started >= 3 * 3600:\n            assert ordinal + 1 < 12, "CPU receiver relay limit"\n            name = f"hung-cpu-metrics-coordinator-{ordinal + 1:02d}-20261027-v1"\n            submit(\n                name,\n                relay_source(bundle, records, index, ordinal + 1, submitted),\n                gpu=False,\n                datasets=[],\n            )\n            with (root / "handoff.json").open("x", encoding="utf-8") as stream:\n                json.dump(\n                    {"next": name, "reader_index": index, "current_reader_submitted": submitted},\n                    stream,\n                )\n            return\n        time.sleep(60)\n'
assert hashlib.sha256(METRIC_SOURCE.encode()).hexdigest()=='e00f050e0d7e6635f723aa4d03d9b1e5391d0da470f758b8dfcce1b28698ab62'
RECEIVER_SOURCE='"""Restore one completed Development shard, verify receipts and score on CPU."""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nimport zipfile\nfrom pathlib import Path\n\n\ndef require(value: bool, message: str) -> None:\n    if not value:\n        raise ValueError(message)\n\n\ndef verification_command(plan: Path, partition: Path, shard_id: str, sha256) -> list[str]:\n    """The receiver has no stage-execution command or execution fallback."""\n    return [\n        sys.executable,\n        "-u",\n        "-m",\n        "experiments.hung_sprint2.dispatch_matched",\n        "--plan",\n        str(plan),\n        "--plan-sha256",\n        sha256(plan),\n        "--shards",\n        str(partition),\n        "--shards-sha256",\n        sha256(partition),\n        "--shard-id",\n        shard_id,\n        "--resume",\n        "--verify-only",\n    ]\n\n\ndef restore(\n    provider,\n    archive: Path,\n    receipt_path: Path,\n    plan_path: Path,\n    partition_path: Path,\n    *,\n    root: Path,\n    expected_code: str,\n) -> dict:\n    """Validate the archive and exact runtime plan before creating any output."""\n    plan = json.loads(plan_path.read_bytes())\n    partition = json.loads(partition_path.read_bytes())\n    receipt = json.loads(receipt_path.read_bytes())\n    require(\n        plan["scope"] == "development_only" and plan["code_commit"] == expected_code,\n        "Development code identity differs",\n    )\n    plan_sha, partition_sha = provider.sha256(plan_path), provider.sha256(partition_path)\n    require(partition["plan_sha256"] == plan_sha == receipt["plan_sha256"], "Plan SHA differs")\n    require(partition_sha == receipt["shards_sha256"], "Partition SHA differs")\n    require(\n        receipt["exit_code"] == receipt["local_verify_exit_code"] == 0, "Execution/verifier failed"\n    )\n    require(receipt["crc"] == "passed", "Original archive CRC failed")\n    shard = next(s for s in partition["shards"] if s["id"] == receipt["shard_id"])\n    jobs = shard["job_ids"]\n    require(0 < len(jobs) <= 20 and len(jobs) == len(set(jobs)), "Not one bounded shard")\n    require(receipt["job_ids"] == jobs, "Receipt job set differs")\n    require(\n        archive.stat().st_size == receipt["artifact_bytes"]\n        and provider.sha256(archive) == receipt["artifact_sha256"],\n        "Archive SHA/size differs",\n    )\n    output = Path(plan["output_root"])\n    require(output.resolve().is_relative_to(root.resolve()), "Output outside receiver root")\n    require(not output.exists(), "Restore output already exists; no overwrite")\n    with zipfile.ZipFile(archive) as stream:\n        names = provider.safe_members(stream)\n        require(\n            json.loads(stream.read("dispatch_identity.json"))["plan_sha256"] == plan_sha,\n            "Archive dispatch identity differs",\n        )\n        require(\n            all(\n                n == "dispatch_identity.json"\n                or (\n                    len(n.split("/")) >= 3\n                    and n.split("/")[0] in {"jobs", "receipts", "logs"}\n                    and n.split("/")[1] in jobs\n                )\n                for n in names\n            ),\n            "Foreign archive member",\n        )\n        output.mkdir(parents=True, exist_ok=False)\n        stream.extractall(output)\n    return receipt\n\n\ndef receive(\n    provider,\n    scorer: dict,\n    archive: Path,\n    receipt_path: Path,\n    repo: Path,\n    *,\n    expected_code: str,\n    evidence: Path,\n    scorer_sha256: str,\n) -> dict:\n    """Verify all official stages and dependencies before recomputing live forecasts."""\n    root = repo.parent\n    require(evidence.resolve().is_relative_to(root.resolve()), "Evidence outside receiver")\n    require(not evidence.exists(), "Receiver evidence already exists")\n    plan = repo / ".cache/provider/plan.json"\n    partition = repo / ".cache/provider/shards.json"\n    started = time.perf_counter()\n    receipt = restore(\n        provider, archive, receipt_path, plan, partition, root=root, expected_code=expected_code\n    )\n    evidence.mkdir(exist_ok=False)\n    command = verification_command(plan, partition, receipt["shard_id"], provider.sha256)\n    environment = {\n        **os.environ,\n        "PYTHONUTF8": "1",\n        "PYTHONPATH": os.pathsep.join((str(repo / "src"), str(repo))),\n    }\n    verify_started = time.perf_counter()\n    with (evidence / "verify.log").open("x", encoding="utf-8") as log:\n        verified = subprocess.run(\n            command,\n            cwd=repo,\n            env=environment,\n            stdout=log,\n            stderr=subprocess.STDOUT,\n            timeout=1800,\n            check=False,\n        )\n    terminal = {\n        "exit_code": verified.returncode,\n        "command": command,\n        "wall_seconds": time.perf_counter() - verify_started,\n        "executed": False,\n        "purpose": "Official --verify-only; no training fallback",\n        "log_sha256": provider.sha256(evidence / "verify.log"),\n    }\n    provider.save(evidence / "verify-exit.json", terminal)\n    require(verified.returncode == 0, "CPU official receipt/dependency verification failed")\n    spec = {\n        "plan": {"path": str(plan), "sha256": provider.sha256(plan)},\n        "partition": {"path": str(partition), "sha256": provider.sha256(partition)},\n        "completed_shards": [\n            {\n                "id": receipt["shard_id"],\n                "verify_exit": {\n                    "path": str(evidence / "verify-exit.json"),\n                    "sha256": provider.sha256(evidence / "verify-exit.json"),\n                },\n            }\n        ],\n    }\n    provider.save(evidence / "collection.json", spec)\n    catalog = scorer["build_catalog"](spec, root)\n    catalog["catalog_sha256"] = provider.sha256(evidence / "collection.json")\n    provider.save(evidence / "catalog.json", catalog)\n    result = scorer["aggregate_results"](catalog, root)\n    result["catalog_sha256"] = provider.sha256(evidence / "catalog.json")\n    provider.save(evidence / "results.json", result)\n    summary = {\n        "schema": "hung-cloud-cpu-receiver-v1",\n        "scope": "development_only",\n        "passed": True,\n        "executed": False,\n        "shard_id": receipt["shard_id"],\n        "jobs": len(receipt["job_ids"]),\n        "counts": result["counts"],\n        "artifact_sha256": receipt["artifact_sha256"],\n        "artifact_bytes": receipt["artifact_bytes"],\n        "original_receipt_sha256": provider.sha256(receipt_path),\n        "model_code_commit": expected_code,\n        "plan_sha256": provider.sha256(plan),\n        "partition_sha256": provider.sha256(partition),\n        "scorer_sha256": scorer_sha256,\n        "verification": terminal,\n        "elapsed_seconds": time.perf_counter() - started,\n        "results_sha256": provider.sha256(evidence / "results.json"),\n        "catalog_sha256": provider.sha256(evidence / "catalog.json"),\n        "peer_acceptance": "pending; technical restore/readback is not independent scientific review",\n    }\n    provider.save(evidence / "readback-summary.json", summary)\n    print(json.dumps(summary), flush=True)\n    return summary\n'
assert hashlib.sha256(RECEIVER_SOURCE.encode()).hexdigest()=='5cfd98dd7c4a64f672a12fce2d5a65454f069d5a2c6a099d3a3d7c7d9fb48571'
SCORER_SOURCE='"""Read-only Development forecast scoring from SHA-pinned, verified artifacts.\n\nRecompute original-unit errors from stored issued forecasts and already arrived\nlabels. Never load Final inputs or choose configurations. All tables are partial\nuntil the separate coverage, fairness and independent-review gates pass.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport struct\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport torch\n\n\ndef file_sha(path: Path) -> str:\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef verified_file(pin: dict, root: Path) -> Path:\n    path = Path(pin["path"]).resolve()\n    if not path.is_relative_to(root.resolve()) or file_sha(path) != pin["sha256"]:\n        raise ValueError("Artifact outside project or SHA mismatch")\n    return path\n\n\ndef pin(path: Path, root: Path) -> dict:\n    path = path.resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError("Artifact outside project")\n    return {"path": str(path), "sha256": file_sha(path)}\n\n\ndef method_family(job: dict) -> str:\n    suffix = "_" + job["cell"]\n    if not job["id"].endswith(suffix) or len(job["id"]) <= len(suffix):\n        raise ValueError("Job family/cell naming differs")\n    return job["id"][: -len(suffix)]\n\n\ndef build_catalog(spec: dict, root: Path) -> dict:\n    """Bind complete dispatcher receipts and VERIFIED_REUSE rows to read-only metrics."""\n    plan = json.loads(verified_file(spec["plan"], root).read_bytes())\n    partition = json.loads(verified_file(spec["partition"], root).read_bytes())\n    if plan["scope"] != "development_only" or partition["plan_sha256"] != spec["plan"]["sha256"]:\n        raise ValueError("Plan/partition binding differs")\n    cells = {c["id"]: c for c in plan["cells"]}\n    jobs = {j["id"]: j for j in plan["jobs"]}\n    configs = {c["id"]: c for c in plan["configs"]}\n    output = Path(spec.get("readback_output_root", plan["output_root"])).resolve()\n    rows = []\n\n    def entry(job_id, method, cell, manifest_pin, checkpoint_pin, source, target, proofs, status):\n        return {\n            "job_id": job_id,\n            "method": method,\n            "budget": cell["budget"],\n            "seed": cell["seed"],\n            "entity_id": cell["entity_id"],\n            "input_manifest_sha256": cell["input_manifest_sha256"],\n            "source_sha256": source["sha256"] if source else None,\n            "target_sha256": target["sha256"] if target else None,\n            "manifest": manifest_pin,\n            "checkpoint": checkpoint_pin,\n            "proofs": proofs,\n            "status": status,\n        }\n\n    if spec.get("reuse_ledger"):\n        ledger = json.loads(verified_file(spec["reuse_ledger"], root).read_bytes())\n        for row in ledger["jobs"]:\n            if row["status"] != "VERIFIED_REUSE":\n                continue\n            if not row["job_id"].startswith("main_"):\n                raise ValueError("Unsupported reuse mapping")\n            cell = cells[row["job_id"][len("main_") :]]\n            manifest_path = verified_file(row["run_manifest"], root)\n            manifest = json.loads(manifest_path.read_bytes())\n            checkpoint = {\n                "path": str(manifest_path.with_name("runner_checkpoint.pt")),\n                "sha256": manifest["final_runner_checkpoint_sha256"],\n            }\n            proofs = [spec["reuse_ledger"], spec["plan"], row["source"], row["target"]]\n            journal = manifest["event_journal"]\n            proofs.append(\n                {"path": str(manifest_path.with_name(journal["file"])), "sha256": journal["sha256"]}\n            )\n            rows.append(\n                entry(\n                    row["job_id"],\n                    "main",\n                    cell,\n                    row["run_manifest"],\n                    checkpoint,\n                    row["source"],\n                    row["target"],\n                    proofs,\n                    "VERIFIED_REUSE",\n                )\n            )\n    for completed in spec["completed_shards"]:\n        exit_record = json.loads(verified_file(completed["verify_exit"], root).read_bytes())\n        if exit_record["exit_code"] != 0:\n            raise ValueError("Shard verifier did not succeed")\n        shard = next(s for s in partition["shards"] if s["id"] == completed["id"])\n        for job_id in shard["job_ids"]:\n            job, cell = jobs[job_id], cells[jobs[job_id]["cell"]]\n            receipt_path = output / "receipts" / job_id / "job.json"\n            receipt = json.loads(receipt_path.read_bytes())\n            if (receipt["job_id"], receipt["plan_sha256"], receipt["shards_sha256"]) != (\n                job_id,\n                spec["plan"]["sha256"],\n                spec["partition"]["sha256"],\n            ):\n                raise ValueError("Job receipt identity differs")\n            config_pin = {k: configs[job["config"]][k] for k in ("path", "sha256")}\n            config = json.loads(verified_file(config_pin, root).read_bytes())\n            final_stage = f"replay-{config[\'replay\'][\'max_origins\']}"\n            proofs = [\n                spec["plan"],\n                spec["partition"],\n                completed["verify_exit"],\n                pin(receipt_path, root),\n                config_pin,\n            ]\n            for stage, sha in receipt["stages"].items():\n                stage_pin = {"path": str(receipt_path.with_name(stage + ".json")), "sha256": sha}\n                stage_record = json.loads(verified_file(stage_pin, root).read_bytes())\n                binding = stage_record["binding"]\n                if (\n                    binding["job"],\n                    binding["plan_sha256"],\n                    binding["shards_sha256"],\n                    binding["config_sha256"],\n                    binding["input_manifest_sha256"],\n                ) != (\n                    job,\n                    spec["plan"]["sha256"],\n                    spec["partition"]["sha256"],\n                    config_pin["sha256"],\n                    cell["input_manifest_sha256"],\n                ):\n                    raise ValueError("Stage provenance differs")\n                proofs.append(stage_pin)\n                for name, record in stage_record["files"].items():\n                    artifact = {\n                        "path": str(output / "jobs" / job_id / stage / name),\n                        "sha256": record["sha256"],\n                    }\n                    verified_file(artifact, root)\n                    proofs.append(artifact)\n            if final_stage not in receipt["stages"]:\n                raise ValueError("Final replay stage incomplete")\n            manifest_path = output / "jobs" / job_id / final_stage / "run_manifest.json"\n            manifest = json.loads(manifest_path.read_bytes())\n            checkpoint_name = (\n                "runner.pt"\n                if manifest["schema"] == "hung-single-baseline-replay-v1"\n                else "runner_checkpoint.pt"\n            )\n            # The final checkpoint hash is already bound by the stage inventory.\n            checkpoint = {\n                "path": str(manifest_path.with_name(checkpoint_name)),\n                "sha256": manifest["final_runner_checkpoint_sha256"],\n            }\n            for role in ("source", "target"):\n                if job[role]:\n                    proofs.append(job[role])\n            rows.append(\n                entry(\n                    job_id,\n                    method_family(job),\n                    cell,\n                    pin(manifest_path, root),\n                    checkpoint,\n                    job["source"],\n                    job["target"],\n                    proofs,\n                    "DISPATCH_VERIFIED",\n                )\n            )\n    return {\n        "scope": "development_only",\n        "schema": "hung-results-catalog-v1",\n        "spec": spec,\n        "runs": rows,\n    }\n\n\ndef metrics(days: list[dict]) -> dict | None:\n    if not days:\n        return None\n    hours = 24 * len(days)\n    return {\n        "hours": hours,\n        "mae_kwh": sum(d["absolute"] for d in days) / hours,\n        "rmse_kwh": math.sqrt(sum(d["squared"] for d in days) / hours),\n    }\n\n\ndef score_saved(state: dict, manifest: dict) -> dict:\n    """Score live predictions only; do not use candidate or audit performance."""\n    if state["schema"] not in {"hung-sprint2-runner-v1", "hung-single-baseline-runner-v1"}:\n        raise ValueError("Unsupported checkpoint schema")\n    if manifest.get("status") != "completed":\n        raise ValueError("Run not terminal")\n    scale = state["energy_scale"]\n    if not math.isfinite(scale) or scale <= 0:\n        raise ValueError("Invalid original-unit scale")\n    days = {}\n    issued_ids = set()\n    for issued in state["issued"]:\n        day_id = issued["day_id"]\n        if day_id in issued_ids:\n            raise ValueError("Duplicate issued day")\n        issued_ids.add(day_id)\n        labels = issued.get("y_scaled")\n        if labels is None:\n            if issued.get("normalized_mae") is not None:\n                raise ValueError("Missing labels carry a score")\n            continue\n        if issued.get("normalized_mae") is None:\n            raise ValueError("Arrived labels were not scored")\n        if state["schema"] == "hung-single-baseline-runner-v1":\n            prediction = issued["prediction"]\n        else:\n            levels = state["controller"]["levels"]\n            prediction = issued["forecasts"][levels.index(issued["active_level"])]\n        prediction, labels = prediction.reshape(-1).double(), labels.reshape(-1).double()\n        if prediction.shape != (24,) or labels.shape != (24,):\n            raise ValueError("Incomplete 24-hour horizon")\n        if not bool(torch.isfinite(prediction).all() and torch.isfinite(labels).all()):\n            raise ValueError("Nonfinite forecast/label")\n        if bool((prediction < 0).any() or (labels < 0).any()):\n            raise ValueError("Negative forecast/label")\n        error = (prediction - labels) * scale\n        days[day_id] = {\n            "absolute": float(error.abs().sum()),\n            "squared": float(error.square().sum()),\n            "label_sha256": hashlib.sha256(\n                struct.pack("<24d", *(labels * scale).tolist())\n            ).hexdigest(),\n        }\n    coverage = manifest["coverage"]\n    if len(issued_ids) != coverage["created"] or len(days) != coverage["scored"]:\n        raise ValueError("Checkpoint coverage differs")\n    calculated = metrics(list(days.values()))\n    declared = manifest["metrics_kwh"]\n    if isinstance(declared, dict) and "live" in declared:\n        declared = declared["live"]\n    if (calculated is None) != (declared is None):\n        raise ValueError("Declared metric availability differs")\n    if calculated is not None:\n        for key, value in calculated.items():\n            if not math.isclose(value, declared[key], rel_tol=1e-9, abs_tol=1e-9):\n                raise ValueError(f"Recomputed {key} differs")\n    return {"coverage": coverage, "metrics": calculated, "days": days}\n\n\ndef verify_artifacts(catalog: dict, root: Path) -> list[dict]:\n    if catalog.get("scope") != "development_only":\n        raise ValueError("Development-only catalog required")\n    records, identities = [], set()\n    for entry in catalog["runs"]:\n        if entry["status"] not in {"VERIFIED_REUSE", "DISPATCH_VERIFIED"}:\n            raise ValueError("Unmatched or unverified run cannot enter metric comparison")\n        identity = (entry["method"], entry["budget"], entry["seed"], entry["entity_id"])\n        if identity in identities:\n            raise ValueError("Duplicate logical run")\n        identities.add(identity)\n        for proof in entry["proofs"]:\n            verified_file(proof, root)\n        manifest = json.loads(verified_file(entry["manifest"], root).read_bytes())\n        checkpoint = verified_file(entry["checkpoint"], root)\n        if manifest["final_runner_checkpoint_sha256"] != entry["checkpoint"]["sha256"]:\n            raise ValueError("Manifest/checkpoint binding differs")\n        binding = manifest["input_binding"]\n        if binding["input_manifest_sha256"] != entry["input_manifest_sha256"]:\n            raise ValueError("Input binding differs")\n        if binding["entity_id"] != entry["entity_id"] or binding["history_days"] != entry["budget"]:\n            raise ValueError("Station/budget binding differs")\n        seed = manifest["config"].get("seed")\n        baseline_receipt_seed = (\n            seed is None\n            and entry["status"] == "DISPATCH_VERIFIED"\n            and entry["method"] in {"seasonal_naive", "frozen_source", "target_only"}\n        )\n        if (seed != entry["seed"] and not baseline_receipt_seed) or not entry["proofs"]:\n            raise ValueError("Seed or verification proof missing")\n        for role in ("source", "target"):\n            expected = entry.get(role + "_sha256")\n            if expected is not None and manifest.get(role + "_checkpoint_sha256") != expected:\n                raise ValueError("Source/Target binding differs")\n        state = torch.load(checkpoint, map_location="cpu", weights_only=True)\n        if state["binding"] != binding:\n            raise ValueError("Checkpoint input binding differs")\n        scored = score_saved(state, manifest)\n        records.append(\n            {\n                **entry,\n                **scored,\n                "code_commit": manifest["code"]["commit"],\n                "device": manifest.get("device"),\n                "final_stage_duration_seconds": manifest.get("duration_seconds"),\n                "cumulative_runtime_costs": manifest.get("runtime_costs", manifest.get("cost")),\n                "cumulative_proxy_costs": manifest.get("proxy_costs"),\n            }\n        )\n    return records\n\n\ndef paired_results(records: list[dict]) -> tuple[list[dict], list[dict]]:\n    cells = defaultdict(dict)\n    for record in records:\n        cells[(record["budget"], record["seed"], record["entity_id"])][record["method"]] = record\n    pairs, blocked = [], []\n    for (budget, seed, station), methods in sorted(cells.items()):\n        for reference in ("main", "target_only"):\n            if reference not in methods:\n                continue\n            baseline = methods[reference]\n            for method, candidate in sorted(methods.items()):\n                if method == reference:\n                    continue\n                reason = None\n                if candidate["input_manifest_sha256"] != baseline["input_manifest_sha256"]:\n                    reason = "input_mismatch"\n                for role in ("source_sha256", "target_sha256"):\n                    left, right = candidate.get(role), baseline.get(role)\n                    if left is not None and right is not None and left != right:\n                        reason = "initialization_mismatch"\n                common = sorted(candidate["days"].keys() & baseline["days"].keys())\n                if any(\n                    candidate["days"][d]["label_sha256"] != baseline["days"][d]["label_sha256"]\n                    for d in common\n                ):\n                    reason = "label_mismatch"\n                if not common:\n                    reason = reason or "no_paired_scored_days"\n                identity = {\n                    "budget": budget,\n                    "seed": seed,\n                    "entity_id": station,\n                    "method": method,\n                    "reference": reference,\n                }\n                if reason:\n                    blocked.append({**identity, "reason": reason})\n                    continue\n                left = metrics([candidate["days"][d] for d in common])\n                right = metrics([baseline["days"][d] for d in common])\n                pairs.append(\n                    {\n                        **identity,\n                        "paired_days": len(common),\n                        "method_metrics": left,\n                        "reference_metrics": right,\n                        "uses_source": candidate.get("source_sha256") is not None,\n                        "paired_day_ids_sha256": hashlib.sha256(\n                            json.dumps(common).encode()\n                        ).hexdigest(),\n                    }\n                )\n    return pairs, blocked\n\n\ndef station_first(pairs: list[dict]) -> list[dict]:\n    stations = defaultdict(list)\n    for pair in pairs:\n        key = (pair["budget"], pair["method"], pair["reference"], pair["entity_id"])\n        stations[key].append(pair)\n    groups = defaultdict(list)\n    for (budget, method, reference, station), rows in sorted(stations.items()):\n        means = {\n            side: {\n                metric: sum(r[side][metric] for r in rows) / len(rows)\n                for metric in ("mae_kwh", "rmse_kwh")\n            }\n            for side in ("method_metrics", "reference_metrics")\n        }\n        groups[(budget, method, reference)].append(\n            {\n                "entity_id": station,\n                "seeds": len(rows),\n                "uses_source": all(r["uses_source"] for r in rows),\n                **means,\n            }\n        )\n    result = []\n    for (budget, method, reference), rows in sorted(groups.items()):\n        averages = {\n            side: {\n                metric: sum(r[side][metric] for r in rows) / len(rows)\n                for metric in ("mae_kwh", "rmse_kwh")\n            }\n            for side in ("method_metrics", "reference_metrics")\n        }\n        gain = averages["reference_metrics"]["mae_kwh"] - averages["method_metrics"]["mae_kwh"]\n        denominator = averages["reference_metrics"]["mae_kwh"]\n        worse = sum(\n            r["method_metrics"]["mae_kwh"] > r["reference_metrics"]["mae_kwh"] for r in rows\n        ) / len(rows)\n        is_transfer = reference == "target_only" and all(r["uses_source"] for r in rows)\n        result.append(\n            {\n                "budget": budget,\n                "method": method,\n                "reference": reference,\n                "stations": len(rows),\n                "minimum_seeds_per_station": min(r["seeds"] for r in rows),\n                **averages,\n                "absolute_mae_gain_kwh": gain,\n                "relative_mae_gain": gain / denominator if denominator > 0 else None,\n                "worse_than_reference_station_fraction": worse,\n                "negative_transfer_station_fraction": worse if is_transfer else None,\n                "transfer_gain": gain / denominator if is_transfer and denominator > 0 else None,\n            }\n        )\n    return result\n\n\ndef aggregate_results(catalog: dict, root: Path) -> dict:\n    records = verify_artifacts(catalog, root)\n    pairs, blocked = paired_results(records)\n    return {\n        "schema": "hung-development-partial-results-v2",\n        "scope": "development_only",\n        "acceptance": "Provisional; full coverage and independent review required",\n        "runs": [{k: v for k, v in r.items() if k != "days"} for r in records],\n        "counts": {\n            "runs": len(records),\n            "scorable_runs": sum(bool(r["days"]) for r in records),\n            "zero_scored_runs": sum(not r["days"] for r in records),\n        },\n        "paired_cells": pairs,\n        "blocked_pairs": blocked,\n        "station_first": station_first(pairs),\n        "aggregation": "Complete hourly errors within paired station/seed; mean seeds within station, then equal station mean within each budget. Budgets remain separate. Transfer gain and negative transfer apply only to Source-using methods versus Target-only. Other comparisons report generic MAE differences/worse-than-reference fractions. Zero reference MAE gives undefined relative gain; absolute differences remain available.",\n    }\n\n\ndef main() -> None:\n    from experiments.hung_sprint2.dispatch_state import atomic_create\n\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--catalog", type=Path, required=True)\n    parser.add_argument("--catalog-sha256", required=True)\n    parser.add_argument("--project-root", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument(\n        "--build-catalog", action="store_true", help="Input is a pinned collection specification"\n    )\n    args = parser.parse_args()\n    path = verified_file(\n        {"path": str(args.catalog), "sha256": args.catalog_sha256}, args.project_root\n    )\n    value = json.loads(path.read_bytes())\n    result = (\n        build_catalog(value, args.project_root)\n        if args.build_catalog\n        else aggregate_results(value, args.project_root)\n    )\n    result["catalog_sha256"] = args.catalog_sha256\n    if not args.output.resolve().is_relative_to(args.project_root.resolve()):\n        raise ValueError("Output outside project")\n    atomic_create(args.output, result)\n    print(\n        json.dumps(\n            {\n                "output": str(args.output),\n                "sha256": file_sha(args.output),\n                **result.get("counts", {}),\n            }\n        )\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
assert hashlib.sha256(SCORER_SOURCE.encode()).hexdigest()=='68fccbed056734d9b4842f7bda088880d7812d12417d64878d1aa018fc010456'
exec(compile(QUEUE_SOURCE,'pinned_queue','exec'),globals())
exec(compile(METRIC_SOURCE,'pinned_metric_queue','exec'),globals())

METRIC_BUNDLE='import hashlib\nQUEUE_SOURCE=\'"""Build bounded private Kaggle Development jobs and a CPU-only relay.\\n\\nUses Kaggle\\\'s existing notebook identity. No credentials, paid scaling, Final\\nlabels or local work are included. Submission failures are never retried.\\n"""\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport time\\nimport zipfile\\nfrom pathlib import Path\\n\\nOWNER = "hungluongduc"\\nPAYLOAD = f"{OWNER}/hung-matched-payload-44b8532-20261027-v1/1"\\nALLOCATION_SHA = "12f5bc9ee402a806faedc89574a4e00c427c384335a3868668adf8ce7f93a0f2"\\nGROUPS = (\\n    (2, 3),\\n    tuple(range(8, 14)),\\n    tuple(range(14, 20)),\\n    tuple(range(20, 26)),\\n    (26, 27, 28),\\n    (33, 34),\\n    (35, 36),\\n)\\nLOCAL = {4, 5, 6, 7, 29, 30, 31, 32, 37, 38, 39}\\nSTORAGE_SOURCE = ""  # Embedded from the versioned storage module when building v2.\\nEXPECTED_JOBSET_SHA = {\\n    "shard_0001": "e2051de08dfb0f0ee91c3617151f08d5cd2c712bb88344244980e4ec2f350c69",\\n    "shard_0002": "1cf89024e9c8b9d8488b386a54a0995924fad19a9334f9539d50843d345260f1",\\n    "shard_0003": "145d75ce5a847f1494d7ec4285e47812c769acb7206bc0da7a20a48e4545f4d8",\\n    "shard_0004": "a7e50150565ca83055e33a9b246e6f8ae7fea6b376d4e1cde6ae55ec0aa8192a",\\n    "shard_0005": "0912c4c7c32350f611bce7493b207cba24810e4aaa20d793a7ee8f2769e1990c",\\n    "shard_0006": "50b870631b6c422dc5b2a29c5e71288a1c07de79428e36cadd989d0a764d1a28",\\n    "shard_0007": "7b448f6b347ad6feae86b5301d97ad4e14a5d590ef70f24c0e6a877149144ec5",\\n    "shard_0008": "928996ffad7ba48f210d2b69cc23cdf4c402a8d34c9a79be76a58a1591d7ff9d",\\n    "shard_0009": "ac51753e9c13f2e6f0677fda3e1e67174fb8a2080a7a859e02465a0c790278f0",\\n    "shard_0010": "9ec71c246ca5334cc5a622cbff9e7b76e6535e447ce0ae6520a8ee9095dc34a3",\\n    "shard_0011": "7ff773c49768e8f73a3b7245e6b6a178de0f599adccee87234f2c4dfa2299932",\\n    "shard_0012": "ae3f9b967f922de325bd35c045eeaafddb3647da47c09d2749887c0351102017",\\n    "shard_0013": "d19ce730e5d2bd6d02c545359838569e1b712269689a58f9ca0e1a03d30b18b8",\\n    "shard_0014": "1454cb90a5e8e0406617e4a0a8a1408040ae570ed6ce9cf81479e08878b5b6a8",\\n    "shard_0015": "e839173725dde0f0e4fb122a9813fac263907e08dd9889d2011073e1e52c30d6",\\n    "shard_0016": "ad6e9fe1219eec06bcde8a95ef333f50bee40c27ed97459375d208e76a01ed0f",\\n    "shard_0017": "a494de718f33adde057a68ef1b4e9e0ea31b56461594f9316fdb6e45211fee75",\\n    "shard_0018": "05c45a5b8bed8e143d16053d892c8abede23633e6c7d96d41288b601f429df20",\\n    "shard_0019": "15761f42e1b901f0b99f98d90a61f9c9f81ba058535054f612be11867048bfef",\\n    "shard_0020": "15e41973e7f727fdcbe7f239922ad02dc05c8efe77cc9609245e29f6278be493",\\n    "shard_0021": "14e4fc0a128f8ed81e52b34d15af9a123503e0d5903f65eb16b5a20c477de5cb",\\n    "shard_0022": "229dfba032263815967bfa18f3a5baef677747ed0c6f828e89202c12703e78fc",\\n    "shard_0023": "f9def5fc4236ed9d9a540c937d554795b4da684d6b0705225dc8dff0f7f01c69",\\n    "shard_0024": "86cb50e120c3081676b0e75a13055659ab17c7092ea8473e311fadad1328ce00",\\n    "shard_0025": "ffc9c9a46dc12a5f843182e1e092c2e434cab2432d218ae1076b987c3b3e1937",\\n    "shard_0026": "3cc709041d096465692797a24d8290c6b8d2e14f3e994d50a734c8fe34e64646",\\n    "shard_0027": "19f14ba77bf687edcf8847933ea5807006096d9a8eb0128560ad20f1cddc5369",\\n    "shard_0028": "a1328e31335db38f13e20e843025c25e6767ed1e86e11ff39b1da10a994db567",\\n    "shard_0029": "33c10aa43b9d1c2675b4cc89e087a12e6c8875c5cd0ac5789a39775a760e7dfc",\\n    "shard_0030": "a908b9eb43e288809722c1c8300370bf0acac961aaaba25d98a169008457395a",\\n    "shard_0031": "35d10c724dd8ac4ed4bb03f864c35288f9edc453900024e12383a328d0c6a0d9",\\n    "shard_0032": "b6fc0d6d0924f2c970da1e40783c6f9ef740d5898458c7caffb7c66830659270",\\n    "shard_0033": "a570c5e215f60b6463e4641f73ce13f7035552162a851d754f524d754031debe",\\n    "shard_0034": "79b8e55f8a74e34b504a736555e9add02e1422bbd5a872509bdab00ae4a21d8a",\\n    "shard_0035": "0ecdc975791fe6b62b0d1be6514d5ca243b3892bf8e8fe612be00dd71a93dac0",\\n    "shard_0036": "c0f24e5038e361798c83d44bd292ef8d099cf575ba00fcab92d55cf178873c2a",\\n    "shard_0037": "aee8e665e66576f461c3a69de62ede64c0840c1189c8e26b51c158cd3b2303ce",\\n    "shard_0038": "09b5c62ae11c1563ee12d6d97c9e10902a48da50a89d80129b581721b35606f2",\\n    "shard_0039": "4177fe39acd6ac291e38b82aea9cc148376f578a58de1bf2d875ed9fa8dddab4",\\n}\\n\\n\\nEXPECTED_PROVIDER_PINS = {\\n    "shard_0002": (\\n        "70dff1c41380aa781062470b6986829e470e320a1cd0522ac49c63956ea6d12d",\\n        "7d99e722c6e27b7257a709182a0827b63c4a1599cfe14f3472e9e4a914cf8ad1",\\n    ),\\n    "shard_0003": (\\n        "70dff1c41380aa781062470b6986829e470e320a1cd0522ac49c63956ea6d12d",\\n        "7d99e722c6e27b7257a709182a0827b63c4a1599cfe14f3472e9e4a914cf8ad1",\\n    ),\\n    "shard_0008": (\\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\\n    ),\\n    "shard_0009": (\\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\\n    ),\\n    "shard_0010": (\\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\\n    ),\\n    "shard_0011": (\\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\\n    ),\\n    "shard_0012": (\\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\\n    ),\\n    "shard_0013": (\\n        "34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4",\\n        "3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342",\\n    ),\\n    "shard_0014": (\\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\\n    ),\\n    "shard_0015": (\\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\\n    ),\\n    "shard_0016": (\\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\\n    ),\\n    "shard_0017": (\\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\\n    ),\\n    "shard_0018": (\\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\\n    ),\\n    "shard_0019": (\\n        "754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d",\\n        "60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781",\\n    ),\\n    "shard_0020": (\\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\\n    ),\\n    "shard_0021": (\\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\\n    ),\\n    "shard_0022": (\\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\\n    ),\\n    "shard_0023": (\\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\\n    ),\\n    "shard_0024": (\\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\\n    ),\\n    "shard_0025": (\\n        "4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5",\\n        "fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666",\\n    ),\\n    "shard_0026": (\\n        "2e9faad307c29ca681eedcc0f64da2ab6ec00ef3893a280d1aead08b30d16836",\\n        "7750cfc08778c747122a2a2e82190a63c5e2f034d68f690748290c599b7c41be",\\n    ),\\n    "shard_0027": (\\n        "2e9faad307c29ca681eedcc0f64da2ab6ec00ef3893a280d1aead08b30d16836",\\n        "7750cfc08778c747122a2a2e82190a63c5e2f034d68f690748290c599b7c41be",\\n    ),\\n    "shard_0028": (\\n        "2e9faad307c29ca681eedcc0f64da2ab6ec00ef3893a280d1aead08b30d16836",\\n        "7750cfc08778c747122a2a2e82190a63c5e2f034d68f690748290c599b7c41be",\\n    ),\\n    "shard_0033": (\\n        "a15728a0c6e54de080e0fbfce2b49fbae9cfeedeaa2a76f281f584f369847886",\\n        "0a02b5c6737445748fcff41d88b2a44cacdfd1f394d80f0a341de47bd983e08b",\\n    ),\\n    "shard_0034": (\\n        "a15728a0c6e54de080e0fbfce2b49fbae9cfeedeaa2a76f281f584f369847886",\\n        "0a02b5c6737445748fcff41d88b2a44cacdfd1f394d80f0a341de47bd983e08b",\\n    ),\\n    "shard_0035": (\\n        "19129a69120e4f1cfd7dc6eae08d9a5ce4a20373fd9f48f584d8faf04801cb7d",\\n        "bd9bc74c3c417f04fbb16781f9c3fb33b07b78d193145dfeae953e7ac610a01f",\\n    ),\\n    "shard_0036": (\\n        "19129a69120e4f1cfd7dc6eae08d9a5ce4a20373fd9f48f584d8faf04801cb7d",\\n        "bd9bc74c3c417f04fbb16781f9c3fb33b07b78d193145dfeae953e7ac610a01f",\\n    ),\\n}\\n\\n\\ndef digest(path: Path) -> str:\\n    result = hashlib.sha256()\\n    with path.open("rb") as stream:\\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\\n            result.update(chunk)\\n    return result.hexdigest()\\n\\n\\ndef validate_groups() -> None:\\n    flat = [n for group in GROUPS for n in group]\\n    assert len(flat) == len(set(flat)) and not set(flat) & LOCAL\\n    assert set(flat) | {1} == {1, 2, 3, *range(8, 29), 33, 34, 35, 36}\\n    assert all(0 < len(group) <= 6 for group in GROUPS)\\n\\n\\ndef slug(index: int) -> str:\\n    assert 0 <= index < len(GROUPS)\\n    return f"hung-matched-group{index + 2:02d}-44b8532-20261027" + ("-v2" if index else "")\\n\\n\\ndef verify_predecessor(\\n    root: Path, shards: tuple[int, ...], *, archive_name="artifacts.zip"\\n) -> list[dict]:\\n    """Hash/CRC readback of a pinned predecessor version, before new training."""\\n    records = []\\n    for number in shards:\\n        name = f"shard_{number:04d}"\\n        matches = [\\n            p\\n            for p in root.rglob("receipt.json")\\n            if json.loads(p.read_text()).get("shard_id") == name\\n            and p.with_name(archive_name).is_file()\\n        ]\\n        assert len(matches) == 1, f"Missing/ambiguous predecessor {name}"\\n        receipt = json.loads(matches[0].read_text())\\n        archive = matches[0].with_name(archive_name)\\n        assert receipt["shard_id"] == name\\n        assert receipt["exit_code"] == receipt["local_verify_exit_code"] == 0\\n        assert (receipt["plan_sha256"], receipt["shards_sha256"]) == EXPECTED_PROVIDER_PINS[name]\\n        jobset = json.dumps(sorted(receipt["job_ids"]), separators=(",", ":")).encode()\\n        assert hashlib.sha256(jobset).hexdigest() == EXPECTED_JOBSET_SHA[name]\\n        assert archive.stat().st_size == receipt["artifact_bytes"]\\n        assert digest(archive) == receipt["artifact_sha256"]\\n        with zipfile.ZipFile(archive) as contents:\\n            names = contents.namelist()\\n            assert len(names) == len(set(names))\\n            assert all(not n.startswith(("/", "\\\\\\\\")) and ".." not in n.split("/") for n in names)\\n            assert contents.testzip() is None\\n            identity = json.loads(contents.read("dispatch_identity.json"))\\n            assert identity["plan_sha256"] == receipt["plan_sha256"]\\n            measured = {\\n                n.split("/")[1]\\n                for n in names\\n                if n.startswith("jobs/") and n.endswith("/dispatch_measurement.json")\\n            }\\n            assert set(measured) == set(receipt["job_ids"])\\n            assert len(receipt["job_ids"]) == len(set(receipt["job_ids"])) == 20\\n        records.append(\\n            {\\n                "shard": name,\\n                "sha256": receipt["artifact_sha256"],\\n                "bytes": receipt["artifact_bytes"],\\n                "crc": "passed",\\n            }\\n        )\\n    return records\\n\\n\\ndef status(slug_name: str) -> str:\\n    from kagglehub.clients import build_kaggle_client\\n    from kagglesdk.kernels.types.kernels_api_service import ApiGetKernelSessionStatusRequest\\n\\n    request = ApiGetKernelSessionStatusRequest()\\n    request.user_name, request.kernel_slug = OWNER, slug_name\\n    with build_kaggle_client() as client:\\n        response = client.kernels.kernels_api_client.get_kernel_session_status(request)\\n    value = response.status.name\\n    assert value in {\\n        "QUEUED",\\n        "RUNNING",\\n        "COMPLETE",\\n        "ERROR",\\n        "NEW_SCRIPT",\\n        "CANCEL_REQUESTED",\\n        "CANCEL_ACKNOWLEDGED",\\n    }, value\\n    return value\\n\\n\\ndef submit(\\n    name: str,\\n    source: str,\\n    *,\\n    gpu: bool,\\n    predecessor: str | None = None,\\n    datasets: list[str] | None = None,\\n) -> dict:\\n    """One submission attempt. Caller must hold the unique relay chain position."""\\n    from kagglehub.clients import build_kaggle_client\\n    from kagglesdk.kernels.types.kernels_api_service import ApiSaveKernelRequest\\n\\n    claim(name + "-submit", {"source_sha256": hashlib.sha256(source.encode()).hexdigest()})\\n    request = ApiSaveKernelRequest()\\n    request.slug, request.new_title = f"{OWNER}/{name}", name\\n    request.text, request.language, request.kernel_type = source, "python", "script"\\n    request.is_private, request.enable_gpu, request.enable_internet = True, gpu, True\\n    request.session_timeout_seconds = 8 * 3600 if gpu else 5 * 3600\\n    request.dataset_data_sources = (\\n        datasets if datasets is not None else ([PAYLOAD] if gpu or predecessor else [])\\n    )\\n    request.kernel_data_sources = [f"{OWNER}/{predecessor}/1"] if predecessor else []\\n    if gpu:\\n        request.machine_shape = "NvidiaTeslaT4"\\n    with build_kaggle_client() as client:\\n        response = client.kernels.kernels_api_client.save_kernel(request)\\n    assert not response.error and response.version_number == 1, "Unexpected version; stop"\\n    assert not response.invalid_dataset_sources and not response.invalid_kernel_sources\\n    record = {\\n        "name": name,\\n        "url": response.url,\\n        "version": response.version_number,\\n        "gpu": gpu,\\n        "source_sha256": hashlib.sha256(source.encode()).hexdigest(),\\n    }\\n    print(json.dumps({"submitted": record}), flush=True)\\n    return record\\n\\n\\ndef claim(name: str, record: dict) -> None:\\n    """Create-only private Dataset claim; 409 never becomes a new version."""\\n    from kagglehub.clients import build_kaggle_client\\n    from kagglehub.gcs_upload import upload_files_and_directories\\n    from kagglesdk.blobs.types.blob_api_service import ApiBlobType\\n    from kagglesdk.datasets.types.dataset_api_service import ApiCreateDatasetRequest\\n\\n    folder = Path("/kaggle/working/claims") / name\\n    folder.mkdir(parents=True, exist_ok=False)\\n    (folder / "claim.json").open("x").write(\\n        json.dumps({**record, "allocation_sha256": ALLOCATION_SHA})\\n    )\\n    upload = upload_files_and_directories(\\n        str(folder), item_type=ApiBlobType.DATASET, ignore_patterns=[]\\n    ).to_proto()\\n    request = ApiCreateDatasetRequest()\\n    request.owner_slug, request.slug, request.title = OWNER, name, name\\n    request.files, request.directories, request.is_private = upload.files, upload.directories, True\\n    with build_kaggle_client() as client:\\n        result = client.datasets.dataset_api_client.create_dataset(request)\\n    assert not result.error, "Claim failed; no submission/training retry"\\n\\n\\ndef module_prefix(module_source: str) -> str:\\n    compile(module_source, "hung_kaggle_queue.py", "exec")\\n    return module_source + "\\\\nMODULE_SOURCE = " + repr(module_source) + "\\\\n"\\n\\n\\ndef compose_module_source(queue_source: str, storage_source: str) -> str:\\n    compile(storage_source, "hung_kaggle_storage.py", "exec")\\n    return queue_source + "\\\\nSTORAGE_SOURCE = " + repr(storage_source) + "\\\\n"\\n\\n\\ndef gpu_source(module_source: str, setup: str, parity: str, index: int) -> str:\\n    assert 1 <= index < len(GROUPS)\\n    selected = [f"shard_{n:04d}" for n in GROUPS[index]]\\n    base_name = f"hung-matched-group{index + 2:02d}-v1"\\n    setup = setup.replace("hung-matched-group02-v1", base_name)\\n    setup = setup.replace(\\\'["shard_0002", "shard_0003"]\\\', repr(selected))\\n    source = module_prefix(module_source)\\n    source += "\\\\nassert STORAGE_SOURCE\\\\nexec(STORAGE_SOURCE, globals())\\\\n"\\n    source += f"\\\\nclaim({slug(index)!r}+\\\'-execute\\\', {{\\\'group_index\\\': {index}}})\\\\n" + setup\\n    source += "\\\\nreadback_started = time.perf_counter()\\\\n"\\n    source += f"readback = verify_predecessor(Path(\\\'/kaggle/input\\\'), {GROUPS[index - 1]!r})\\\\n"\\n    source += "provider.save(base / \\\'predecessor-readback.json\\\', {\\\'records\\\': readback, \\\'seconds\\\': time.perf_counter()-readback_started})\\\\n"\\n    source += "print(json.dumps({\\\'predecessor_readback\\\': readback}), flush=True)\\\\n"\\n    source += "\\\\n" + parity + "\\\\nassert passed\\\\n"\\n    source += f"selected = {selected!r}\\\\nreceipts = []\\\\ngroup_start = time.perf_counter()\\\\n"\\n    source += """\\nfor shard_id in selected:\\n    from kagglehub.clients import build_kaggle_client\\n    from kagglesdk.kernels.types.kernels_api_service import ApiGetAcceleratorQuotaStatisticsRequest\\n    with build_kaggle_client() as client:\\n        quota = client.kernels.kernels_api_client.get_accelerator_quota_statistics(\\n            ApiGetAcceleratorQuotaStatisticsRequest()).gpu_quota\\n    assert quota is not None and not quota.is_pay_to_scale_enabled\\n    assert all(v is not None for v in (quota.total_time_allowed, quota.time_used, quota.time_reserved))\\n    available = (quota.total_time_allowed-quota.time_used-quota.time_reserved).total_seconds()\\n    assert available >= 3600 + 20 * 1810.4 / 20 * 1.25, \\\'Fresh quota + reserve failed\\\'\\n    assert shutil.disk_usage(base).free >= 8*1024**3, \\\'Disk reserve failed\\\'\\n    assert time.perf_counter()-group_start < 6*3600, \\\'Bounded session limit\\\'\\n    record = provider.execute_shard(repo, shard_id, base / shard_id)\\n    persist_shard(record, base/shard_id/\\\'artifacts.zip\\\', module_prefix(MODULE_SOURCE),\\n                  owner=OWNER, submit=submit, status=status)\\n    receipts.append(record)\\n    print(json.dumps(record), flush=True)\\nprovider.save(base / \\\'group-complete.json\\\', {\\\'shards\\\': selected, \\\'receipts\\\': receipts,\\n    \\\'allocation_queue_sha256\\\': ALLOCATION_SHA, \\\'scope\\\': \\\'Development only; peer acceptance pending\\\'})\\nprint(json.dumps({\\\'group_complete\\\': True, \\\'shards\\\': selected}), flush=True)\\n"""\\n    compile(source, base_name, "exec")\\n    return source\\n\\n\\ndef relay_source(\\n    module_source: str, setup: str, parity: str, index: int, ordinal: int, *, submit_current: bool\\n) -> str:\\n    assert 0 <= index < len(GROUPS) and ordinal < 12\\n    return (\\n        module_prefix(module_source)\\n        + "\\\\nSETUP = "\\n        + repr(setup)\\n        + "\\\\nPARITY = "\\n        + repr(parity)\\n        + f"\\\\nrelay(MODULE_SOURCE, SETUP, PARITY, {index}, {ordinal}, "\\n        + f"submit_current={submit_current!r})\\\\n"\\n    )\\n\\n\\ndef relay(\\n    module_source: str, setup: str, parity: str, index: int, ordinal: int, *, submit_current: bool\\n) -> None:\\n    """CPU SDK relay: serial fixed GPU groups; 3h CPU relay bounds, no LLM polling."""\\n    validate_groups()\\n    claim(f"hung-development-relay-{ordinal:02d}-execute-20261027-v2", {"group_index": index})\\n    evidence = Path(f"/kaggle/working/relay-{ordinal:02d}")\\n    evidence.mkdir(exist_ok=False)\\n    started = time.monotonic()\\n    if submit_current:\\n        assert index > 0\\n        submit(\\n            slug(index),\\n            gpu_source(module_source, setup, parity, index),\\n            gpu=True,\\n            predecessor=slug(index - 1),\\n        )\\n    while True:\\n        state = status(slug(index))\\n        assert state not in {"ERROR", "CANCEL_REQUESTED", "CANCEL_ACKNOWLEDGED", "NEW_SCRIPT"}, (\\n            state\\n        )\\n        if state == "COMPLETE":\\n            (evidence / f"group-{index:02d}-terminal.json").open("x").write(\\n                json.dumps({"status": state, "version": 1, "name": slug(index)})\\n            )\\n            if index == len(GROUPS) - 1:\\n                # A separate CPU version mounts the last completed Output and checks bytes.\\n                proof = (\\n                    module_prefix(module_source)\\n                    + f"\\\\nrecords=verify_predecessor(Path(\\\'/kaggle/input\\\'), {GROUPS[index]!r})\\\\n"\\n                    + "Path(\\\'/kaggle/working/final-development-readback.json\\\').write_text(json.dumps({\\\'records\\\':records,\\\'scope\\\':\\\'Development technical readback only; no Final evaluation\\\'}))\\\\n"\\n                    + "print(json.dumps({\\\'passed\\\':True,\\\'records\\\':records}),flush=True)\\\\n"\\n                )\\n                submit(\\n                    "hung-development-last-readback-20261027-v2",\\n                    proof,\\n                    gpu=False,\\n                    predecessor=slug(index),\\n                )\\n                return\\n            index += 1\\n            submit(\\n                slug(index),\\n                gpu_source(module_source, setup, parity, index),\\n                gpu=True,\\n                predecessor=slug(index - 1),\\n            )\\n        if time.monotonic() - started >= 3 * 3600:\\n            assert ordinal + 1 < 12, "CPU relay count bound"\\n            name = f"hung-development-cpu-relay-{ordinal + 1:02d}-20261027-v2"\\n            submit(\\n                name,\\n                relay_source(\\n                    module_source, setup, parity, index, ordinal + 1, submit_current=False\\n                ),\\n                gpu=False,\\n            )\\n            (evidence / "handoff.json").open("x").write(json.dumps({"next": name, "index": index}))\\n            return\\n        time.sleep(60)\\n\'\nassert hashlib.sha256(QUEUE_SOURCE.encode()).hexdigest()==\'4ee0718f077e587e14619ec2effe1637de93a18750d81da11e8c576a6de63093\'\nMETRIC_SOURCE=\'"""Finite CPU-only receivers for the existing immutable Kaggle Development route."""\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport time\\nfrom pathlib import Path\\n\\nTRIAL = "hung-shard0008-full-cpu-restore-20261027-v1"\\nMODEL_CODE = "44b8532959d76fb20691881539f991ca44983727"\\nPAYLOAD_SHA = "23a6d420523d990d07ff07d814ed642a851925283a102a3ae806f3b666e1d6f4"\\nHELPER_SHA = "cadcd80d671d3b49a317312f125735362ffba3068247f1714a0ae56c6be83424"\\n\\n\\ndef bundle_sources(queue: str, metric_queue: str, receiver: str, scorer: str) -> str:\\n    """Compile separate versioned modules, preserving future-import boundaries."""\\n    sources = {\\n        "QUEUE_SOURCE": queue,\\n        "METRIC_SOURCE": metric_queue,\\n        "RECEIVER_SOURCE": receiver,\\n        "SCORER_SOURCE": scorer,\\n    }\\n    result = "import hashlib\\\\n"\\n    for name, source in sources.items():\\n        compile(source, name, "exec")\\n        result += f"{name}={source!r}\\\\n"\\n        result += f"assert hashlib.sha256({name}.encode()).hexdigest()=={hashlib.sha256(source.encode()).hexdigest()!r}\\\\n"\\n    result += "exec(compile(QUEUE_SOURCE,\\\'pinned_queue\\\',\\\'exec\\\'),globals())\\\\n"\\n    result += "exec(compile(METRIC_SOURCE,\\\'pinned_metric_queue\\\',\\\'exec\\\'),globals())\\\\n"\\n    compile(result, "cpu-metric-bundle", "exec")\\n    return result\\n\\n\\ndef readers(groups: tuple, provider_pins: dict) -> list[dict]:\\n    """Only allocated Kaggle shards; the already-running shard8 trial is reused."""\\n    result = [\\n        {\\n            "shard_id": "shard_0001",\\n            "training": "hung-matched-shard0001-44b8532-20261027",\\n            "base": "hung-matched-shard0001-v1",\\n            "plan_sha256": "80a1666cf2083594c9a2bc0b0a4dc3e30118eda43d07187cd5b6d8b51f90e0bb",\\n            "partition_sha256": "e2c6cb6f15a7473c6cdfd7912fefa838d63a770cf352dff93c2cccc3ae86a15c",\\n        }\\n    ]\\n    for index, group in enumerate(groups):\\n        training = (\\n            "hung-matched-group03-recovery-20261027-v2"\\n            if index == 1\\n            else f"hung-matched-group{index + 2:02d}-44b8532-20261027" + ("-v2" if index else "")\\n        )\\n        for number in group:\\n            if number == 8:\\n                continue\\n            shard = f"shard_{number:04d}"\\n            plan_sha, partition_sha = provider_pins[shard]\\n            result.append(\\n                {\\n                    "shard_id": shard,\\n                    "training": training,\\n                    "base": f"hung-matched-group{index + 2:02d}-v1",\\n                    "plan_sha256": plan_sha,\\n                    "partition_sha256": partition_sha,\\n                }\\n            )\\n    expected = {1, 2, 3, *range(9, 29), 33, 34, 35, 36}\\n    assert {int(r["shard_id"][6:]) for r in result} == expected and len(result) == len(expected)\\n    return result\\n\\n\\ndef reader_name(record: dict) -> str:\\n    return f"hung-cpu-metrics-{record[\\\'shard_id\\\'].replace(\\\'_\\\', \\\'\\\')}-20261027-v1"\\n\\n\\ndef owned_kernel_exists(slug_name: str) -> bool:\\n    """Observe owned resources before asking status of a not-yet-submitted group."""\\n    from kagglehub.clients import build_kaggle_client\\n    from kagglesdk.kernels.types.kernels_api_service import ApiListKernelsRequest\\n    from kagglesdk.kernels.types.kernels_enums import KernelsListViewType\\n\\n    request = ApiListKernelsRequest()\\n    request.user, request.page_size = "hungluongduc", 100\\n    request.group = KernelsListViewType.PUBLIC_AND_USERS_PRIVATE\\n    seen = set()\\n    with build_kaggle_client() as client:\\n        for _ in range(10):\\n            result = client.kernels.kernels_api_client.list_kernels(request)\\n            if any(k.ref == f"hungluongduc/{slug_name}" for k in result.kernels):\\n                return True\\n            if not result.next_page_token:\\n                return False\\n            assert result.next_page_token not in seen, "Owned inventory pagination repeated"\\n            seen.add(result.next_page_token)\\n            request.page_token = result.next_page_token\\n    raise ValueError("Owned inventory bound exceeded; no status/submission inference")\\n\\n\\ndef receiver_source(record: dict, receiver: str, scorer: str) -> str:\\n    import hashlib\\n\\n    source = """import hashlib, importlib.util, json, types\\nfrom pathlib import Path\\nimport torch\\nassert not torch.cuda.is_available(), \\\'CPU receiver only\\\'\\n"""\\n    source += f"SPEC={record!r}\\\\nRECEIVER_SOURCE={receiver!r}\\\\nSCORER_SOURCE={scorer!r}\\\\n"\\n    source += f"MODEL_CODE={MODEL_CODE!r}\\\\nPAYLOAD_SHA={PAYLOAD_SHA!r}\\\\nHELPER_SHA={HELPER_SHA!r}\\\\n"\\n    source += f"RECEIVER_SHA={hashlib.sha256(receiver.encode()).hexdigest()!r}\\\\n"\\n    source += f"SCORER_SHA={hashlib.sha256(scorer.encode()).hexdigest()!r}\\\\n"\\n    source += """\\npayloads=list(Path(\\\'/kaggle/input\\\').rglob(\\\'provider-payload.zip.bin\\\'))\\nassert len(payloads)==1, \\\'Pinned payload missing/ambiguous\\\'\\nhelper=payloads[0].parent/\\\'hung_provider.py\\\'\\nassert hashlib.sha256(helper.read_bytes()).hexdigest()==HELPER_SHA\\nmodule_spec=importlib.util.spec_from_file_location(\\\'hung_provider\\\',helper)\\nprovider=importlib.util.module_from_spec(module_spec); module_spec.loader.exec_module(provider)\\nreceipts=[p for p in Path(\\\'/kaggle/input\\\').rglob(\\\'receipt.json\\\')\\n          if p.with_name(\\\'artifacts.zip\\\').is_file()\\n          and json.loads(p.read_bytes()).get(\\\'shard_id\\\')==SPEC[\\\'shard_id\\\']]\\nassert len(receipts)==1, \\\'Canonical completed shard missing/ambiguous\\\'\\nrepo,plan=provider.materialize(payloads[0],PAYLOAD_SHA,Path(\\\'/kaggle/working\\\')/SPEC[\\\'base\\\'])\\nassert provider.sha256(plan)==SPEC[\\\'plan_sha256\\\']\\nassert provider.sha256(repo/\\\'.cache/provider/shards.json\\\')==SPEC[\\\'partition_sha256\\\']\\nassert hashlib.sha256(RECEIVER_SOURCE.encode()).hexdigest()==RECEIVER_SHA\\nassert hashlib.sha256(SCORER_SOURCE.encode()).hexdigest()==SCORER_SHA\\nreceiver=types.ModuleType(\\\'pinned_cloud_receiver\\\')\\nexec(compile(RECEIVER_SOURCE,\\\'pinned_cloud_receiver\\\',\\\'exec\\\'),receiver.__dict__)\\nscorer={\\\'__name__\\\':\\\'pinned_saved_forecast_scorer\\\'}\\nexec(compile(SCORER_SOURCE,\\\'pinned_saved_forecast_scorer\\\',\\\'exec\\\'),scorer)\\nsummary=receiver.receive(provider,scorer,receipts[0].with_name(\\\'artifacts.zip\\\'),receipts[0],repo,\\n                         expected_code=MODEL_CODE,evidence=repo.parent/\\\'cpu-readback\\\',\\n                         scorer_sha256=SCORER_SHA)\\nassert summary[\\\'passed\\\'] and summary[\\\'executed\\\'] is False and summary[\\\'jobs\\\']==20\\nsummary.update({\\\'receiver_sha256\\\':RECEIVER_SHA,\\\'transport_helper_sha256\\\':HELPER_SHA,\\n                \\\'derivation\\\':json.loads((repo.parent/\\\'derivation.json\\\').read_bytes()),\\n                \\\'device\\\':\\\'cpu\\\',\\\'torch\\\':torch.__version__})\\nassert summary[\\\'derivation\\\'][\\\'parent_plan_sha256\\\']==\\\'2d448d2584f65baf7e09445e013b4283314b08ea2cce855c6850e9fc12ba87ee\\\'\\nprovider.save(repo.parent/\\\'receiver-proof.json\\\',summary)\\nprint(json.dumps({\\\'receiver_proof\\\':summary}),flush=True)\\n"""\\n    compile(source, "cpu-shard-receiver", "exec")\\n    return source\\n\\n\\ndef relay_source(\\n    bundle: str, records: list[dict], index: int, ordinal: int, submitted: bool\\n) -> str:\\n    source = bundle + f"\\\\nMETRIC_BUNDLE={bundle!r}\\\\nMETRIC_READERS={records!r}\\\\n"\\n    source += (\\n        "assert owned_kernel_exists(METRIC_READERS[0][\\\'training\\\']), \\\'Owned inventory capability gate failed\\\'\\\\n"\\n        "print(json.dumps({\\\'owned_inventory_capability\\\':\\\'passed\\\',\\\'gpu\\\':False}),flush=True)\\\\n"\\n    )\\n    source += (\\n        f"run_queue(METRIC_BUNDLE,METRIC_READERS,{index},{ordinal},{submitted!r},"\\n        "submit=submit,status=status,claim=claim,exists=owned_kernel_exists,receiver=RECEIVER_SOURCE,scorer=SCORER_SOURCE)\\\\n"\\n    )\\n    compile(source, "cpu-receiver-relay", "exec")\\n    return source\\n\\n\\ndef run_queue(\\n    bundle: str,\\n    records: list[dict],\\n    index: int,\\n    ordinal: int,\\n    submitted: bool,\\n    *,\\n    submit,\\n    status,\\n    claim,\\n    exists,\\n    receiver: str,\\n    scorer: str,\\n) -> None:\\n    """Only provider status handles drive handoffs; no GPU or submission retry."""\\n    assert 0 <= index < len(records) and ordinal < 12\\n    assert len({r["shard_id"] for r in records}) == len(records)\\n    claim(f"hung-cpu-metrics-relay-{ordinal:02d}-execute-20261027-v1", {"reader_index": index})\\n    root = Path(f"/kaggle/working/cpu-metrics-relay-{ordinal:02d}")\\n    root.mkdir(exist_ok=False)\\n    started = time.monotonic()\\n    known_training = set()\\n    while True:\\n        trial = status(TRIAL)\\n        assert trial in {"QUEUED", "RUNNING", "COMPLETE"}, f"CPU trial failed: {trial}"\\n        record = records[index]\\n        if trial == "COMPLETE":\\n            if not submitted:\\n                name = record["training"]\\n                if name not in known_training and exists(name):\\n                    known_training.add(name)\\n                training = status(name) if name in known_training else "NOT_OBSERVED"\\n                assert training in {"NOT_OBSERVED", "QUEUED", "RUNNING", "COMPLETE"}, (\\n                    f"Provider failed: {training}"\\n                )\\n                if training == "COMPLETE":\\n                    submit(\\n                        reader_name(record),\\n                        receiver_source(record, receiver, scorer),\\n                        gpu=False,\\n                        predecessor=record["training"],\\n                    )\\n                    submitted = True\\n            if submitted:\\n                state = status(reader_name(record))\\n                assert state in {"QUEUED", "RUNNING", "COMPLETE"}, f"CPU receiver failed: {state}"\\n                if state == "COMPLETE":\\n                    proof = {\\n                        "shard_id": record["shard_id"],\\n                        "cpu_reader": reader_name(record),\\n                        "version": 1,\\n                        "status": state,\\n                        "scope": "Development technical readback only",\\n                    }\\n                    with (root / (record["shard_id"] + ".json")).open(\\n                        "x", encoding="utf-8"\\n                    ) as stream:\\n                        json.dump(proof, stream)\\n                    print(json.dumps({"completed_cpu_receiver": proof}), flush=True)\\n                    index += 1\\n                    submitted = False\\n                    if index == len(records):\\n                        with (root / "complete.json").open("x", encoding="utf-8") as stream:\\n                            json.dump({"receivers": len(records)}, stream)\\n                        return\\n        if time.monotonic() - started >= 3 * 3600:\\n            assert ordinal + 1 < 12, "CPU receiver relay limit"\\n            name = f"hung-cpu-metrics-coordinator-{ordinal + 1:02d}-20261027-v1"\\n            submit(\\n                name,\\n                relay_source(bundle, records, index, ordinal + 1, submitted),\\n                gpu=False,\\n                datasets=[],\\n            )\\n            with (root / "handoff.json").open("x", encoding="utf-8") as stream:\\n                json.dump(\\n                    {"next": name, "reader_index": index, "current_reader_submitted": submitted},\\n                    stream,\\n                )\\n            return\\n        time.sleep(60)\\n\'\nassert hashlib.sha256(METRIC_SOURCE.encode()).hexdigest()==\'e00f050e0d7e6635f723aa4d03d9b1e5391d0da470f758b8dfcce1b28698ab62\'\nRECEIVER_SOURCE=\'"""Restore one completed Development shard, verify receipts and score on CPU."""\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nimport subprocess\\nimport sys\\nimport time\\nimport zipfile\\nfrom pathlib import Path\\n\\n\\ndef require(value: bool, message: str) -> None:\\n    if not value:\\n        raise ValueError(message)\\n\\n\\ndef verification_command(plan: Path, partition: Path, shard_id: str, sha256) -> list[str]:\\n    """The receiver has no stage-execution command or execution fallback."""\\n    return [\\n        sys.executable,\\n        "-u",\\n        "-m",\\n        "experiments.hung_sprint2.dispatch_matched",\\n        "--plan",\\n        str(plan),\\n        "--plan-sha256",\\n        sha256(plan),\\n        "--shards",\\n        str(partition),\\n        "--shards-sha256",\\n        sha256(partition),\\n        "--shard-id",\\n        shard_id,\\n        "--resume",\\n        "--verify-only",\\n    ]\\n\\n\\ndef restore(\\n    provider,\\n    archive: Path,\\n    receipt_path: Path,\\n    plan_path: Path,\\n    partition_path: Path,\\n    *,\\n    root: Path,\\n    expected_code: str,\\n) -> dict:\\n    """Validate the archive and exact runtime plan before creating any output."""\\n    plan = json.loads(plan_path.read_bytes())\\n    partition = json.loads(partition_path.read_bytes())\\n    receipt = json.loads(receipt_path.read_bytes())\\n    require(\\n        plan["scope"] == "development_only" and plan["code_commit"] == expected_code,\\n        "Development code identity differs",\\n    )\\n    plan_sha, partition_sha = provider.sha256(plan_path), provider.sha256(partition_path)\\n    require(partition["plan_sha256"] == plan_sha == receipt["plan_sha256"], "Plan SHA differs")\\n    require(partition_sha == receipt["shards_sha256"], "Partition SHA differs")\\n    require(\\n        receipt["exit_code"] == receipt["local_verify_exit_code"] == 0, "Execution/verifier failed"\\n    )\\n    require(receipt["crc"] == "passed", "Original archive CRC failed")\\n    shard = next(s for s in partition["shards"] if s["id"] == receipt["shard_id"])\\n    jobs = shard["job_ids"]\\n    require(0 < len(jobs) <= 20 and len(jobs) == len(set(jobs)), "Not one bounded shard")\\n    require(receipt["job_ids"] == jobs, "Receipt job set differs")\\n    require(\\n        archive.stat().st_size == receipt["artifact_bytes"]\\n        and provider.sha256(archive) == receipt["artifact_sha256"],\\n        "Archive SHA/size differs",\\n    )\\n    output = Path(plan["output_root"])\\n    require(output.resolve().is_relative_to(root.resolve()), "Output outside receiver root")\\n    require(not output.exists(), "Restore output already exists; no overwrite")\\n    with zipfile.ZipFile(archive) as stream:\\n        names = provider.safe_members(stream)\\n        require(\\n            json.loads(stream.read("dispatch_identity.json"))["plan_sha256"] == plan_sha,\\n            "Archive dispatch identity differs",\\n        )\\n        require(\\n            all(\\n                n == "dispatch_identity.json"\\n                or (\\n                    len(n.split("/")) >= 3\\n                    and n.split("/")[0] in {"jobs", "receipts", "logs"}\\n                    and n.split("/")[1] in jobs\\n                )\\n                for n in names\\n            ),\\n            "Foreign archive member",\\n        )\\n        output.mkdir(parents=True, exist_ok=False)\\n        stream.extractall(output)\\n    return receipt\\n\\n\\ndef receive(\\n    provider,\\n    scorer: dict,\\n    archive: Path,\\n    receipt_path: Path,\\n    repo: Path,\\n    *,\\n    expected_code: str,\\n    evidence: Path,\\n    scorer_sha256: str,\\n) -> dict:\\n    """Verify all official stages and dependencies before recomputing live forecasts."""\\n    root = repo.parent\\n    require(evidence.resolve().is_relative_to(root.resolve()), "Evidence outside receiver")\\n    require(not evidence.exists(), "Receiver evidence already exists")\\n    plan = repo / ".cache/provider/plan.json"\\n    partition = repo / ".cache/provider/shards.json"\\n    started = time.perf_counter()\\n    receipt = restore(\\n        provider, archive, receipt_path, plan, partition, root=root, expected_code=expected_code\\n    )\\n    evidence.mkdir(exist_ok=False)\\n    command = verification_command(plan, partition, receipt["shard_id"], provider.sha256)\\n    environment = {\\n        **os.environ,\\n        "PYTHONUTF8": "1",\\n        "PYTHONPATH": os.pathsep.join((str(repo / "src"), str(repo))),\\n    }\\n    verify_started = time.perf_counter()\\n    with (evidence / "verify.log").open("x", encoding="utf-8") as log:\\n        verified = subprocess.run(\\n            command,\\n            cwd=repo,\\n            env=environment,\\n            stdout=log,\\n            stderr=subprocess.STDOUT,\\n            timeout=1800,\\n            check=False,\\n        )\\n    terminal = {\\n        "exit_code": verified.returncode,\\n        "command": command,\\n        "wall_seconds": time.perf_counter() - verify_started,\\n        "executed": False,\\n        "purpose": "Official --verify-only; no training fallback",\\n        "log_sha256": provider.sha256(evidence / "verify.log"),\\n    }\\n    provider.save(evidence / "verify-exit.json", terminal)\\n    require(verified.returncode == 0, "CPU official receipt/dependency verification failed")\\n    spec = {\\n        "plan": {"path": str(plan), "sha256": provider.sha256(plan)},\\n        "partition": {"path": str(partition), "sha256": provider.sha256(partition)},\\n        "completed_shards": [\\n            {\\n                "id": receipt["shard_id"],\\n                "verify_exit": {\\n                    "path": str(evidence / "verify-exit.json"),\\n                    "sha256": provider.sha256(evidence / "verify-exit.json"),\\n                },\\n            }\\n        ],\\n    }\\n    provider.save(evidence / "collection.json", spec)\\n    catalog = scorer["build_catalog"](spec, root)\\n    catalog["catalog_sha256"] = provider.sha256(evidence / "collection.json")\\n    provider.save(evidence / "catalog.json", catalog)\\n    result = scorer["aggregate_results"](catalog, root)\\n    result["catalog_sha256"] = provider.sha256(evidence / "catalog.json")\\n    provider.save(evidence / "results.json", result)\\n    summary = {\\n        "schema": "hung-cloud-cpu-receiver-v1",\\n        "scope": "development_only",\\n        "passed": True,\\n        "executed": False,\\n        "shard_id": receipt["shard_id"],\\n        "jobs": len(receipt["job_ids"]),\\n        "counts": result["counts"],\\n        "artifact_sha256": receipt["artifact_sha256"],\\n        "artifact_bytes": receipt["artifact_bytes"],\\n        "original_receipt_sha256": provider.sha256(receipt_path),\\n        "model_code_commit": expected_code,\\n        "plan_sha256": provider.sha256(plan),\\n        "partition_sha256": provider.sha256(partition),\\n        "scorer_sha256": scorer_sha256,\\n        "verification": terminal,\\n        "elapsed_seconds": time.perf_counter() - started,\\n        "results_sha256": provider.sha256(evidence / "results.json"),\\n        "catalog_sha256": provider.sha256(evidence / "catalog.json"),\\n        "peer_acceptance": "pending; technical restore/readback is not independent scientific review",\\n    }\\n    provider.save(evidence / "readback-summary.json", summary)\\n    print(json.dumps(summary), flush=True)\\n    return summary\\n\'\nassert hashlib.sha256(RECEIVER_SOURCE.encode()).hexdigest()==\'5cfd98dd7c4a64f672a12fce2d5a65454f069d5a2c6a099d3a3d7c7d9fb48571\'\nSCORER_SOURCE=\'"""Read-only Development forecast scoring from SHA-pinned, verified artifacts.\\n\\nRecompute original-unit errors from stored issued forecasts and already arrived\\nlabels. Never load Final inputs or choose configurations. All tables are partial\\nuntil the separate coverage, fairness and independent-review gates pass.\\n"""\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport json\\nimport math\\nimport struct\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport torch\\n\\n\\ndef file_sha(path: Path) -> str:\\n    with path.open("rb") as stream:\\n        return hashlib.file_digest(stream, "sha256").hexdigest()\\n\\n\\ndef verified_file(pin: dict, root: Path) -> Path:\\n    path = Path(pin["path"]).resolve()\\n    if not path.is_relative_to(root.resolve()) or file_sha(path) != pin["sha256"]:\\n        raise ValueError("Artifact outside project or SHA mismatch")\\n    return path\\n\\n\\ndef pin(path: Path, root: Path) -> dict:\\n    path = path.resolve()\\n    if not path.is_relative_to(root.resolve()):\\n        raise ValueError("Artifact outside project")\\n    return {"path": str(path), "sha256": file_sha(path)}\\n\\n\\ndef method_family(job: dict) -> str:\\n    suffix = "_" + job["cell"]\\n    if not job["id"].endswith(suffix) or len(job["id"]) <= len(suffix):\\n        raise ValueError("Job family/cell naming differs")\\n    return job["id"][: -len(suffix)]\\n\\n\\ndef build_catalog(spec: dict, root: Path) -> dict:\\n    """Bind complete dispatcher receipts and VERIFIED_REUSE rows to read-only metrics."""\\n    plan = json.loads(verified_file(spec["plan"], root).read_bytes())\\n    partition = json.loads(verified_file(spec["partition"], root).read_bytes())\\n    if plan["scope"] != "development_only" or partition["plan_sha256"] != spec["plan"]["sha256"]:\\n        raise ValueError("Plan/partition binding differs")\\n    cells = {c["id"]: c for c in plan["cells"]}\\n    jobs = {j["id"]: j for j in plan["jobs"]}\\n    configs = {c["id"]: c for c in plan["configs"]}\\n    output = Path(spec.get("readback_output_root", plan["output_root"])).resolve()\\n    rows = []\\n\\n    def entry(job_id, method, cell, manifest_pin, checkpoint_pin, source, target, proofs, status):\\n        return {\\n            "job_id": job_id,\\n            "method": method,\\n            "budget": cell["budget"],\\n            "seed": cell["seed"],\\n            "entity_id": cell["entity_id"],\\n            "input_manifest_sha256": cell["input_manifest_sha256"],\\n            "source_sha256": source["sha256"] if source else None,\\n            "target_sha256": target["sha256"] if target else None,\\n            "manifest": manifest_pin,\\n            "checkpoint": checkpoint_pin,\\n            "proofs": proofs,\\n            "status": status,\\n        }\\n\\n    if spec.get("reuse_ledger"):\\n        ledger = json.loads(verified_file(spec["reuse_ledger"], root).read_bytes())\\n        for row in ledger["jobs"]:\\n            if row["status"] != "VERIFIED_REUSE":\\n                continue\\n            if not row["job_id"].startswith("main_"):\\n                raise ValueError("Unsupported reuse mapping")\\n            cell = cells[row["job_id"][len("main_") :]]\\n            manifest_path = verified_file(row["run_manifest"], root)\\n            manifest = json.loads(manifest_path.read_bytes())\\n            checkpoint = {\\n                "path": str(manifest_path.with_name("runner_checkpoint.pt")),\\n                "sha256": manifest["final_runner_checkpoint_sha256"],\\n            }\\n            proofs = [spec["reuse_ledger"], spec["plan"], row["source"], row["target"]]\\n            journal = manifest["event_journal"]\\n            proofs.append(\\n                {"path": str(manifest_path.with_name(journal["file"])), "sha256": journal["sha256"]}\\n            )\\n            rows.append(\\n                entry(\\n                    row["job_id"],\\n                    "main",\\n                    cell,\\n                    row["run_manifest"],\\n                    checkpoint,\\n                    row["source"],\\n                    row["target"],\\n                    proofs,\\n                    "VERIFIED_REUSE",\\n                )\\n            )\\n    for completed in spec["completed_shards"]:\\n        exit_record = json.loads(verified_file(completed["verify_exit"], root).read_bytes())\\n        if exit_record["exit_code"] != 0:\\n            raise ValueError("Shard verifier did not succeed")\\n        shard = next(s for s in partition["shards"] if s["id"] == completed["id"])\\n        for job_id in shard["job_ids"]:\\n            job, cell = jobs[job_id], cells[jobs[job_id]["cell"]]\\n            receipt_path = output / "receipts" / job_id / "job.json"\\n            receipt = json.loads(receipt_path.read_bytes())\\n            if (receipt["job_id"], receipt["plan_sha256"], receipt["shards_sha256"]) != (\\n                job_id,\\n                spec["plan"]["sha256"],\\n                spec["partition"]["sha256"],\\n            ):\\n                raise ValueError("Job receipt identity differs")\\n            config_pin = {k: configs[job["config"]][k] for k in ("path", "sha256")}\\n            config = json.loads(verified_file(config_pin, root).read_bytes())\\n            final_stage = f"replay-{config[\\\'replay\\\'][\\\'max_origins\\\']}"\\n            proofs = [\\n                spec["plan"],\\n                spec["partition"],\\n                completed["verify_exit"],\\n                pin(receipt_path, root),\\n                config_pin,\\n            ]\\n            for stage, sha in receipt["stages"].items():\\n                stage_pin = {"path": str(receipt_path.with_name(stage + ".json")), "sha256": sha}\\n                stage_record = json.loads(verified_file(stage_pin, root).read_bytes())\\n                binding = stage_record["binding"]\\n                if (\\n                    binding["job"],\\n                    binding["plan_sha256"],\\n                    binding["shards_sha256"],\\n                    binding["config_sha256"],\\n                    binding["input_manifest_sha256"],\\n                ) != (\\n                    job,\\n                    spec["plan"]["sha256"],\\n                    spec["partition"]["sha256"],\\n                    config_pin["sha256"],\\n                    cell["input_manifest_sha256"],\\n                ):\\n                    raise ValueError("Stage provenance differs")\\n                proofs.append(stage_pin)\\n                for name, record in stage_record["files"].items():\\n                    artifact = {\\n                        "path": str(output / "jobs" / job_id / stage / name),\\n                        "sha256": record["sha256"],\\n                    }\\n                    verified_file(artifact, root)\\n                    proofs.append(artifact)\\n            if final_stage not in receipt["stages"]:\\n                raise ValueError("Final replay stage incomplete")\\n            manifest_path = output / "jobs" / job_id / final_stage / "run_manifest.json"\\n            manifest = json.loads(manifest_path.read_bytes())\\n            checkpoint_name = (\\n                "runner.pt"\\n                if manifest["schema"] == "hung-single-baseline-replay-v1"\\n                else "runner_checkpoint.pt"\\n            )\\n            # The final checkpoint hash is already bound by the stage inventory.\\n            checkpoint = {\\n                "path": str(manifest_path.with_name(checkpoint_name)),\\n                "sha256": manifest["final_runner_checkpoint_sha256"],\\n            }\\n            for role in ("source", "target"):\\n                if job[role]:\\n                    proofs.append(job[role])\\n            rows.append(\\n                entry(\\n                    job_id,\\n                    method_family(job),\\n                    cell,\\n                    pin(manifest_path, root),\\n                    checkpoint,\\n                    job["source"],\\n                    job["target"],\\n                    proofs,\\n                    "DISPATCH_VERIFIED",\\n                )\\n            )\\n    return {\\n        "scope": "development_only",\\n        "schema": "hung-results-catalog-v1",\\n        "spec": spec,\\n        "runs": rows,\\n    }\\n\\n\\ndef metrics(days: list[dict]) -> dict | None:\\n    if not days:\\n        return None\\n    hours = 24 * len(days)\\n    return {\\n        "hours": hours,\\n        "mae_kwh": sum(d["absolute"] for d in days) / hours,\\n        "rmse_kwh": math.sqrt(sum(d["squared"] for d in days) / hours),\\n    }\\n\\n\\ndef score_saved(state: dict, manifest: dict) -> dict:\\n    """Score live predictions only; do not use candidate or audit performance."""\\n    if state["schema"] not in {"hung-sprint2-runner-v1", "hung-single-baseline-runner-v1"}:\\n        raise ValueError("Unsupported checkpoint schema")\\n    if manifest.get("status") != "completed":\\n        raise ValueError("Run not terminal")\\n    scale = state["energy_scale"]\\n    if not math.isfinite(scale) or scale <= 0:\\n        raise ValueError("Invalid original-unit scale")\\n    days = {}\\n    issued_ids = set()\\n    for issued in state["issued"]:\\n        day_id = issued["day_id"]\\n        if day_id in issued_ids:\\n            raise ValueError("Duplicate issued day")\\n        issued_ids.add(day_id)\\n        labels = issued.get("y_scaled")\\n        if labels is None:\\n            if issued.get("normalized_mae") is not None:\\n                raise ValueError("Missing labels carry a score")\\n            continue\\n        if issued.get("normalized_mae") is None:\\n            raise ValueError("Arrived labels were not scored")\\n        if state["schema"] == "hung-single-baseline-runner-v1":\\n            prediction = issued["prediction"]\\n        else:\\n            levels = state["controller"]["levels"]\\n            prediction = issued["forecasts"][levels.index(issued["active_level"])]\\n        prediction, labels = prediction.reshape(-1).double(), labels.reshape(-1).double()\\n        if prediction.shape != (24,) or labels.shape != (24,):\\n            raise ValueError("Incomplete 24-hour horizon")\\n        if not bool(torch.isfinite(prediction).all() and torch.isfinite(labels).all()):\\n            raise ValueError("Nonfinite forecast/label")\\n        if bool((prediction < 0).any() or (labels < 0).any()):\\n            raise ValueError("Negative forecast/label")\\n        error = (prediction - labels) * scale\\n        days[day_id] = {\\n            "absolute": float(error.abs().sum()),\\n            "squared": float(error.square().sum()),\\n            "label_sha256": hashlib.sha256(\\n                struct.pack("<24d", *(labels * scale).tolist())\\n            ).hexdigest(),\\n        }\\n    coverage = manifest["coverage"]\\n    if len(issued_ids) != coverage["created"] or len(days) != coverage["scored"]:\\n        raise ValueError("Checkpoint coverage differs")\\n    calculated = metrics(list(days.values()))\\n    declared = manifest["metrics_kwh"]\\n    if isinstance(declared, dict) and "live" in declared:\\n        declared = declared["live"]\\n    if (calculated is None) != (declared is None):\\n        raise ValueError("Declared metric availability differs")\\n    if calculated is not None:\\n        for key, value in calculated.items():\\n            if not math.isclose(value, declared[key], rel_tol=1e-9, abs_tol=1e-9):\\n                raise ValueError(f"Recomputed {key} differs")\\n    return {"coverage": coverage, "metrics": calculated, "days": days}\\n\\n\\ndef verify_artifacts(catalog: dict, root: Path) -> list[dict]:\\n    if catalog.get("scope") != "development_only":\\n        raise ValueError("Development-only catalog required")\\n    records, identities = [], set()\\n    for entry in catalog["runs"]:\\n        if entry["status"] not in {"VERIFIED_REUSE", "DISPATCH_VERIFIED"}:\\n            raise ValueError("Unmatched or unverified run cannot enter metric comparison")\\n        identity = (entry["method"], entry["budget"], entry["seed"], entry["entity_id"])\\n        if identity in identities:\\n            raise ValueError("Duplicate logical run")\\n        identities.add(identity)\\n        for proof in entry["proofs"]:\\n            verified_file(proof, root)\\n        manifest = json.loads(verified_file(entry["manifest"], root).read_bytes())\\n        checkpoint = verified_file(entry["checkpoint"], root)\\n        if manifest["final_runner_checkpoint_sha256"] != entry["checkpoint"]["sha256"]:\\n            raise ValueError("Manifest/checkpoint binding differs")\\n        binding = manifest["input_binding"]\\n        if binding["input_manifest_sha256"] != entry["input_manifest_sha256"]:\\n            raise ValueError("Input binding differs")\\n        if binding["entity_id"] != entry["entity_id"] or binding["history_days"] != entry["budget"]:\\n            raise ValueError("Station/budget binding differs")\\n        seed = manifest["config"].get("seed")\\n        baseline_receipt_seed = (\\n            seed is None\\n            and entry["status"] == "DISPATCH_VERIFIED"\\n            and entry["method"] in {"seasonal_naive", "frozen_source", "target_only"}\\n        )\\n        if (seed != entry["seed"] and not baseline_receipt_seed) or not entry["proofs"]:\\n            raise ValueError("Seed or verification proof missing")\\n        for role in ("source", "target"):\\n            expected = entry.get(role + "_sha256")\\n            if expected is not None and manifest.get(role + "_checkpoint_sha256") != expected:\\n                raise ValueError("Source/Target binding differs")\\n        state = torch.load(checkpoint, map_location="cpu", weights_only=True)\\n        if state["binding"] != binding:\\n            raise ValueError("Checkpoint input binding differs")\\n        scored = score_saved(state, manifest)\\n        records.append(\\n            {\\n                **entry,\\n                **scored,\\n                "code_commit": manifest["code"]["commit"],\\n                "device": manifest.get("device"),\\n                "final_stage_duration_seconds": manifest.get("duration_seconds"),\\n                "cumulative_runtime_costs": manifest.get("runtime_costs", manifest.get("cost")),\\n                "cumulative_proxy_costs": manifest.get("proxy_costs"),\\n            }\\n        )\\n    return records\\n\\n\\ndef paired_results(records: list[dict]) -> tuple[list[dict], list[dict]]:\\n    cells = defaultdict(dict)\\n    for record in records:\\n        cells[(record["budget"], record["seed"], record["entity_id"])][record["method"]] = record\\n    pairs, blocked = [], []\\n    for (budget, seed, station), methods in sorted(cells.items()):\\n        for reference in ("main", "target_only"):\\n            if reference not in methods:\\n                continue\\n            baseline = methods[reference]\\n            for method, candidate in sorted(methods.items()):\\n                if method == reference:\\n                    continue\\n                reason = None\\n                if candidate["input_manifest_sha256"] != baseline["input_manifest_sha256"]:\\n                    reason = "input_mismatch"\\n                for role in ("source_sha256", "target_sha256"):\\n                    left, right = candidate.get(role), baseline.get(role)\\n                    if left is not None and right is not None and left != right:\\n                        reason = "initialization_mismatch"\\n                common = sorted(candidate["days"].keys() & baseline["days"].keys())\\n                if any(\\n                    candidate["days"][d]["label_sha256"] != baseline["days"][d]["label_sha256"]\\n                    for d in common\\n                ):\\n                    reason = "label_mismatch"\\n                if not common:\\n                    reason = reason or "no_paired_scored_days"\\n                identity = {\\n                    "budget": budget,\\n                    "seed": seed,\\n                    "entity_id": station,\\n                    "method": method,\\n                    "reference": reference,\\n                }\\n                if reason:\\n                    blocked.append({**identity, "reason": reason})\\n                    continue\\n                left = metrics([candidate["days"][d] for d in common])\\n                right = metrics([baseline["days"][d] for d in common])\\n                pairs.append(\\n                    {\\n                        **identity,\\n                        "paired_days": len(common),\\n                        "method_metrics": left,\\n                        "reference_metrics": right,\\n                        "uses_source": candidate.get("source_sha256") is not None,\\n                        "paired_day_ids_sha256": hashlib.sha256(\\n                            json.dumps(common).encode()\\n                        ).hexdigest(),\\n                    }\\n                )\\n    return pairs, blocked\\n\\n\\ndef station_first(pairs: list[dict]) -> list[dict]:\\n    stations = defaultdict(list)\\n    for pair in pairs:\\n        key = (pair["budget"], pair["method"], pair["reference"], pair["entity_id"])\\n        stations[key].append(pair)\\n    groups = defaultdict(list)\\n    for (budget, method, reference, station), rows in sorted(stations.items()):\\n        means = {\\n            side: {\\n                metric: sum(r[side][metric] for r in rows) / len(rows)\\n                for metric in ("mae_kwh", "rmse_kwh")\\n            }\\n            for side in ("method_metrics", "reference_metrics")\\n        }\\n        groups[(budget, method, reference)].append(\\n            {\\n                "entity_id": station,\\n                "seeds": len(rows),\\n                "uses_source": all(r["uses_source"] for r in rows),\\n                **means,\\n            }\\n        )\\n    result = []\\n    for (budget, method, reference), rows in sorted(groups.items()):\\n        averages = {\\n            side: {\\n                metric: sum(r[side][metric] for r in rows) / len(rows)\\n                for metric in ("mae_kwh", "rmse_kwh")\\n            }\\n            for side in ("method_metrics", "reference_metrics")\\n        }\\n        gain = averages["reference_metrics"]["mae_kwh"] - averages["method_metrics"]["mae_kwh"]\\n        denominator = averages["reference_metrics"]["mae_kwh"]\\n        worse = sum(\\n            r["method_metrics"]["mae_kwh"] > r["reference_metrics"]["mae_kwh"] for r in rows\\n        ) / len(rows)\\n        is_transfer = reference == "target_only" and all(r["uses_source"] for r in rows)\\n        result.append(\\n            {\\n                "budget": budget,\\n                "method": method,\\n                "reference": reference,\\n                "stations": len(rows),\\n                "minimum_seeds_per_station": min(r["seeds"] for r in rows),\\n                **averages,\\n                "absolute_mae_gain_kwh": gain,\\n                "relative_mae_gain": gain / denominator if denominator > 0 else None,\\n                "worse_than_reference_station_fraction": worse,\\n                "negative_transfer_station_fraction": worse if is_transfer else None,\\n                "transfer_gain": gain / denominator if is_transfer and denominator > 0 else None,\\n            }\\n        )\\n    return result\\n\\n\\ndef aggregate_results(catalog: dict, root: Path) -> dict:\\n    records = verify_artifacts(catalog, root)\\n    pairs, blocked = paired_results(records)\\n    return {\\n        "schema": "hung-development-partial-results-v2",\\n        "scope": "development_only",\\n        "acceptance": "Provisional; full coverage and independent review required",\\n        "runs": [{k: v for k, v in r.items() if k != "days"} for r in records],\\n        "counts": {\\n            "runs": len(records),\\n            "scorable_runs": sum(bool(r["days"]) for r in records),\\n            "zero_scored_runs": sum(not r["days"] for r in records),\\n        },\\n        "paired_cells": pairs,\\n        "blocked_pairs": blocked,\\n        "station_first": station_first(pairs),\\n        "aggregation": "Complete hourly errors within paired station/seed; mean seeds within station, then equal station mean within each budget. Budgets remain separate. Transfer gain and negative transfer apply only to Source-using methods versus Target-only. Other comparisons report generic MAE differences/worse-than-reference fractions. Zero reference MAE gives undefined relative gain; absolute differences remain available.",\\n    }\\n\\n\\ndef main() -> None:\\n    from experiments.hung_sprint2.dispatch_state import atomic_create\\n\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--catalog", type=Path, required=True)\\n    parser.add_argument("--catalog-sha256", required=True)\\n    parser.add_argument("--project-root", type=Path, required=True)\\n    parser.add_argument("--output", type=Path, required=True)\\n    parser.add_argument(\\n        "--build-catalog", action="store_true", help="Input is a pinned collection specification"\\n    )\\n    args = parser.parse_args()\\n    path = verified_file(\\n        {"path": str(args.catalog), "sha256": args.catalog_sha256}, args.project_root\\n    )\\n    value = json.loads(path.read_bytes())\\n    result = (\\n        build_catalog(value, args.project_root)\\n        if args.build_catalog\\n        else aggregate_results(value, args.project_root)\\n    )\\n    result["catalog_sha256"] = args.catalog_sha256\\n    if not args.output.resolve().is_relative_to(args.project_root.resolve()):\\n        raise ValueError("Output outside project")\\n    atomic_create(args.output, result)\\n    print(\\n        json.dumps(\\n            {\\n                "output": str(args.output),\\n                "sha256": file_sha(args.output),\\n                **result.get("counts", {}),\\n            }\\n        )\\n    )\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\'\nassert hashlib.sha256(SCORER_SOURCE.encode()).hexdigest()==\'68fccbed056734d9b4842f7bda088880d7812d12417d64878d1aa018fc010456\'\nexec(compile(QUEUE_SOURCE,\'pinned_queue\',\'exec\'),globals())\nexec(compile(METRIC_SOURCE,\'pinned_metric_queue\',\'exec\'),globals())\n'
METRIC_READERS=[{'shard_id': 'shard_0001', 'training': 'hung-matched-shard0001-44b8532-20261027', 'base': 'hung-matched-shard0001-v1', 'plan_sha256': '80a1666cf2083594c9a2bc0b0a4dc3e30118eda43d07187cd5b6d8b51f90e0bb', 'partition_sha256': 'e2c6cb6f15a7473c6cdfd7912fefa838d63a770cf352dff93c2cccc3ae86a15c'}, {'shard_id': 'shard_0002', 'training': 'hung-matched-group02-44b8532-20261027', 'base': 'hung-matched-group02-v1', 'plan_sha256': '70dff1c41380aa781062470b6986829e470e320a1cd0522ac49c63956ea6d12d', 'partition_sha256': '7d99e722c6e27b7257a709182a0827b63c4a1599cfe14f3472e9e4a914cf8ad1'}, {'shard_id': 'shard_0003', 'training': 'hung-matched-group02-44b8532-20261027', 'base': 'hung-matched-group02-v1', 'plan_sha256': '70dff1c41380aa781062470b6986829e470e320a1cd0522ac49c63956ea6d12d', 'partition_sha256': '7d99e722c6e27b7257a709182a0827b63c4a1599cfe14f3472e9e4a914cf8ad1'}, {'shard_id': 'shard_0009', 'training': 'hung-matched-group03-recovery-20261027-v2', 'base': 'hung-matched-group03-v1', 'plan_sha256': '34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4', 'partition_sha256': '3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342'}, {'shard_id': 'shard_0010', 'training': 'hung-matched-group03-recovery-20261027-v2', 'base': 'hung-matched-group03-v1', 'plan_sha256': '34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4', 'partition_sha256': '3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342'}, {'shard_id': 'shard_0011', 'training': 'hung-matched-group03-recovery-20261027-v2', 'base': 'hung-matched-group03-v1', 'plan_sha256': '34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4', 'partition_sha256': '3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342'}, {'shard_id': 'shard_0012', 'training': 'hung-matched-group03-recovery-20261027-v2', 'base': 'hung-matched-group03-v1', 'plan_sha256': '34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4', 'partition_sha256': '3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342'}, {'shard_id': 'shard_0013', 'training': 'hung-matched-group03-recovery-20261027-v2', 'base': 'hung-matched-group03-v1', 'plan_sha256': '34f252fd19e33698a56faf4f3b19e921248ddf419b91fdfc3a69e7bc3f63f1a4', 'partition_sha256': '3f022fe12fe9918062959240b8f1dd3be9b25f18fdcae982ba972f867e78c342'}, {'shard_id': 'shard_0014', 'training': 'hung-matched-group04-44b8532-20261027-v2', 'base': 'hung-matched-group04-v1', 'plan_sha256': '754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d', 'partition_sha256': '60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781'}, {'shard_id': 'shard_0015', 'training': 'hung-matched-group04-44b8532-20261027-v2', 'base': 'hung-matched-group04-v1', 'plan_sha256': '754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d', 'partition_sha256': '60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781'}, {'shard_id': 'shard_0016', 'training': 'hung-matched-group04-44b8532-20261027-v2', 'base': 'hung-matched-group04-v1', 'plan_sha256': '754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d', 'partition_sha256': '60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781'}, {'shard_id': 'shard_0017', 'training': 'hung-matched-group04-44b8532-20261027-v2', 'base': 'hung-matched-group04-v1', 'plan_sha256': '754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d', 'partition_sha256': '60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781'}, {'shard_id': 'shard_0018', 'training': 'hung-matched-group04-44b8532-20261027-v2', 'base': 'hung-matched-group04-v1', 'plan_sha256': '754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d', 'partition_sha256': '60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781'}, {'shard_id': 'shard_0019', 'training': 'hung-matched-group04-44b8532-20261027-v2', 'base': 'hung-matched-group04-v1', 'plan_sha256': '754fa4f3a7e79a017b64fb36ca19624c224f84b773cb88c570f74cd406b7ea4d', 'partition_sha256': '60b67243dd5911dd0a63061ad7430732db3381bdc0777c14b900316fc46e6781'}, {'shard_id': 'shard_0020', 'training': 'hung-matched-group05-44b8532-20261027-v2', 'base': 'hung-matched-group05-v1', 'plan_sha256': '4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5', 'partition_sha256': 'fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666'}, {'shard_id': 'shard_0021', 'training': 'hung-matched-group05-44b8532-20261027-v2', 'base': 'hung-matched-group05-v1', 'plan_sha256': '4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5', 'partition_sha256': 'fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666'}, {'shard_id': 'shard_0022', 'training': 'hung-matched-group05-44b8532-20261027-v2', 'base': 'hung-matched-group05-v1', 'plan_sha256': '4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5', 'partition_sha256': 'fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666'}, {'shard_id': 'shard_0023', 'training': 'hung-matched-group05-44b8532-20261027-v2', 'base': 'hung-matched-group05-v1', 'plan_sha256': '4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5', 'partition_sha256': 'fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666'}, {'shard_id': 'shard_0024', 'training': 'hung-matched-group05-44b8532-20261027-v2', 'base': 'hung-matched-group05-v1', 'plan_sha256': '4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5', 'partition_sha256': 'fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666'}, {'shard_id': 'shard_0025', 'training': 'hung-matched-group05-44b8532-20261027-v2', 'base': 'hung-matched-group05-v1', 'plan_sha256': '4b0c6e5b63d5a7e28165454988650bcbdcc6b61ccea375899840197780574fe5', 'partition_sha256': 'fddd50e4992855dd5519b8562bce1daee0a2dfc31ab20181a23e11a7b5405666'}, {'shard_id': 'shard_0026', 'training': 'hung-matched-group06-44b8532-20261027-v2', 'base': 'hung-matched-group06-v1', 'plan_sha256': '2e9faad307c29ca681eedcc0f64da2ab6ec00ef3893a280d1aead08b30d16836', 'partition_sha256': '7750cfc08778c747122a2a2e82190a63c5e2f034d68f690748290c599b7c41be'}, {'shard_id': 'shard_0027', 'training': 'hung-matched-group06-44b8532-20261027-v2', 'base': 'hung-matched-group06-v1', 'plan_sha256': '2e9faad307c29ca681eedcc0f64da2ab6ec00ef3893a280d1aead08b30d16836', 'partition_sha256': '7750cfc08778c747122a2a2e82190a63c5e2f034d68f690748290c599b7c41be'}, {'shard_id': 'shard_0028', 'training': 'hung-matched-group06-44b8532-20261027-v2', 'base': 'hung-matched-group06-v1', 'plan_sha256': '2e9faad307c29ca681eedcc0f64da2ab6ec00ef3893a280d1aead08b30d16836', 'partition_sha256': '7750cfc08778c747122a2a2e82190a63c5e2f034d68f690748290c599b7c41be'}, {'shard_id': 'shard_0033', 'training': 'hung-matched-group07-44b8532-20261027-v2', 'base': 'hung-matched-group07-v1', 'plan_sha256': 'a15728a0c6e54de080e0fbfce2b49fbae9cfeedeaa2a76f281f584f369847886', 'partition_sha256': '0a02b5c6737445748fcff41d88b2a44cacdfd1f394d80f0a341de47bd983e08b'}, {'shard_id': 'shard_0034', 'training': 'hung-matched-group07-44b8532-20261027-v2', 'base': 'hung-matched-group07-v1', 'plan_sha256': 'a15728a0c6e54de080e0fbfce2b49fbae9cfeedeaa2a76f281f584f369847886', 'partition_sha256': '0a02b5c6737445748fcff41d88b2a44cacdfd1f394d80f0a341de47bd983e08b'}, {'shard_id': 'shard_0035', 'training': 'hung-matched-group08-44b8532-20261027-v2', 'base': 'hung-matched-group08-v1', 'plan_sha256': '19129a69120e4f1cfd7dc6eae08d9a5ce4a20373fd9f48f584d8faf04801cb7d', 'partition_sha256': 'bd9bc74c3c417f04fbb16781f9c3fb33b07b78d193145dfeae953e7ac610a01f'}, {'shard_id': 'shard_0036', 'training': 'hung-matched-group08-44b8532-20261027-v2', 'base': 'hung-matched-group08-v1', 'plan_sha256': '19129a69120e4f1cfd7dc6eae08d9a5ce4a20373fd9f48f584d8faf04801cb7d', 'partition_sha256': 'bd9bc74c3c417f04fbb16781f9c3fb33b07b78d193145dfeae953e7ac610a01f'}]
assert owned_kernel_exists(METRIC_READERS[0]['training']), 'Owned inventory capability gate failed'
print(json.dumps({'owned_inventory_capability':'passed','gpu':False}),flush=True)
run_queue(METRIC_BUNDLE,METRIC_READERS,0,0,False,submit=submit,status=status,claim=claim,exists=owned_kernel_exists,receiver=RECEIVER_SOURCE,scorer=SCORER_SOURCE)
